Phase 1 — BiGRU 
1. DataLoader
   → charger .npy par recording
   → mapper labels via stride_map
   → padding/masking séquences variables
   → class weights

2. Modèle BiGRU Dual-Head
   → Input: 2304d
   → BiGRU 2 couches + dropout
   → Attention pooling
   → Head 1: action classification (72 classes)
   → Head 2: anomaly score (optionnel maintenant)

3. Training loop
   → CrossEntropy pondérée (class weights cap=10)
   → Early stopping
   → F1 macro sur val
   → Target : F1 > 0.40

Phase 2 — Viterbi (après BiGRU stable)
4. Graphe transitions
   → compter transitions train
   → matrice probabilités A[i→j]
   
5. Viterbi decoding
   → post-processing prédictions BiGRU
   → éliminer séquences impossibles
   → minimum segment duration
   → Target : F1 > 0.50
Phase 3 — Mahalanobis (après Viterbi)
6. Prototype Bank
   → mean features par classe (train)
   → matrice covariance inverse

7. Anomaly Score
   → distance Mahalanobis feature → prototype
   → seuils NORMAL/MONITOR/PAUSE/STOP

8. Décision cobot
   → combiner BiGRU + Viterbi + Mahalanobis

In [1]:
import os, pickle, re, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

TRAIN_FEAT_DIR = '/kaggle/input/datasets/hibabou/industrial/data/train/'
VAL_FEAT_DIR   = '/kaggle/input/datasets/hibabou/industrial/data/val/'
PREP_DIR       = '/kaggle/input/datasets/hibabou/prepbigru/kaggle/working/'
FEAT_DIM       = 2304
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

Device : cuda


In [ ]:
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

FEAT_DIR = '/kaggle/input/datasets/hibabou/featuressupp/sf_features_test/'

files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('.npy')])
print(f'Total fichiers : {len(files)}\n')

all_stds, all_cos = [], []
for f in files:
    arr = np.load(os.path.join(FEAT_DIR, f))
    std = arr.std()
    cos = cosine_similarity(arr[:-1], arr[1:]).diagonal().mean()
    all_stds.append(std)
    all_cos.append(cos)
    print(f'  {f} | shape={arr.shape} | std={std:.4f} | cos={cos:.4f}')

print(f'\n── GLOBAL ──────────────────────')
print(f'Total     : {len(files)} fichiers')
print(f'Std moyen : {np.mean(all_stds):.4f}')
print(f'Cos moyen : {np.mean(all_cos):.4f}')
print('✅ Features correctes' if np.mean(all_stds) >= 0.35 else '❌ Re-extraire')

Total fichiers : 32

  test_p1_03_assy_0_1_features.npy | shape=(10896, 2304) | std=0.3970 | cos=0.5793
  test_p1_03_assy_1_3_features.npy | shape=(10400, 2304) | std=0.4107 | cos=0.6031
  test_p1_03_main_0_1_features.npy | shape=(5272, 2304) | std=0.4127 | cos=0.6084
  test_p1_08_assy_0_1_features.npy | shape=(20984, 2304) | std=0.3991 | cos=0.5998
  test_p1_08_assy_2_4_features.npy | shape=(13648, 2304) | std=0.3833 | cos=0.6140
  test_p1_08_main_0_1_features.npy | shape=(10256, 2304) | std=0.3842 | cos=0.5828
  test_p1_09_assy_0_1_features.npy | shape=(13880, 2304) | std=0.3963 | cos=0.6062
  test_p1_09_assy_3_1_features.npy | shape=(15176, 2304) | std=0.4159 | cos=0.6050
  test_p1_09_main_0_1_features.npy | shape=(10096, 2304) | std=0.3927 | cos=0.5808
  test_p2_10_assy_0_1_features.npy | shape=(10224, 2304) | std=0.4070 | cos=0.5874
  test_p2_10_assy_3_2_features.npy | shape=(10264, 2304) | std=0.4021 | cos=0.5743
  test_p2_10_main_0_1_features.npy | shape=(5928, 2304) | std=0.385

In [1]:
import os
import numpy as np

# Chemins de toutes les features
TRAIN_FEAT_DIR = '/kaggle/input/datasets/hibabou/industrial/data/train/'
VAL_FEAT_DIR   = '/kaggle/input/datasets/hibabou/industrial/data/val/'
TEST_FEAT_DIR  = '/kaggle/input/datasets/hibabou/featuressupp/sf_features_test/'

def get_stats(feat_dir, split_name):
    files = sorted([f for f in os.listdir(feat_dir) if f.endswith('.npy')])
    shapes, stds = [], []
    for f in files:
        arr = np.load(os.path.join(feat_dir, f))
        shapes.append(arr.shape[1])  # dimension features
        stds.append(arr.std())
    print(f'\n=== {split_name} ===')
    print(f'Fichiers        : {len(files)}')
    print(f'Feat dim unique : {set(shapes)} (attendu {{2304}})')
    print(f'Std moyen       : {np.mean(stds):.4f}')
    print(f'Std min/max     : {np.min(stds):.4f} / {np.max(stds):.4f}')
    print('✅ Homogène' if set(shapes) == {2304} else '❌ Dimensions différentes')

get_stats(TRAIN_FEAT_DIR, 'TRAIN')
get_stats(VAL_FEAT_DIR,   'VAL')
get_stats(TEST_FEAT_DIR,  'TEST')


=== TRAIN ===
Fichiers        : 36
Feat dim unique : {2304} (attendu {2304})
Std moyen       : 0.4101
Std min/max     : 0.3730 / 0.4599
✅ Homogène

=== VAL ===
Fichiers        : 16
Feat dim unique : {2304} (attendu {2304})
Std moyen       : 0.4016
Std min/max     : 0.3716 / 0.4229
✅ Homogène

=== TEST ===
Fichiers        : 32
Feat dim unique : {2304} (attendu {2304})
Std moyen       : 0.3963
Std min/max     : 0.3601 / 0.4247
✅ Homogène


In [3]:
import os

# Télécharger labels
os.system('curl -L -o /kaggle/working/labels.zip "https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/778d7d5f-6631-4cb6-9451-f88c574c7991"')
os.system('unzip -q /kaggle/working/labels.zip -d /kaggle/working/labels/')

# Vérifier
print(os.listdir('/kaggle/working/labels/'))

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

['val.csv', 'test.csv', 'train.csv']


100 67816  100 67816    0     0   132k      0 --:--:-- --:--:-- --:--:--  132k


In [4]:
import pandas as pd

col_names = ['recording', 'action_id', 'action_name', 'frame_start', 'frame_end']
test_df  = pd.read_csv('/kaggle/working/labels/test.csv',  header=None, names=col_names)
train_df = pd.read_csv('/kaggle/working/labels/train.csv', header=None, names=col_names)
val_df   = pd.read_csv('/kaggle/working/labels/val.csv',   header=None, names=col_names)

for col in ['frame_start', 'frame_end']:
    for df in [train_df, val_df, test_df]:
        df[col] = df[col].str.replace('.jpg','').astype(int)

# Classes originales test
counts = test_df['action_id'].value_counts().sort_index()
total  = len(test_df)

print(f'Total segments test : {total}')
print(f'Classes uniques     : {test_df["action_id"].nunique()}')
print(f'\n{"ID":>4} {"Action":>30} {"Train":>7} {"Val":>7} {"Test":>7}')
print('-' * 60)

train_counts = train_df['action_id'].value_counts()
val_counts   = val_df['action_id'].value_counts()
test_counts  = test_df['action_id'].value_counts()

all_ids = sorted(set(train_df['action_id']) | set(val_df['action_id']) | set(test_df['action_id']))
for aid in all_ids:
    name = test_df[test_df['action_id']==aid]['action_name'].values
    if len(name) == 0:
        name = train_df[train_df['action_id']==aid]['action_name'].values[0]
    else:
        name = name[0]
    t = train_counts.get(aid, 0)
    v = val_counts.get(aid, 0)
    te = test_counts.get(aid, 0)
    print(f'{aid:>4} {name:>30} {t:>7} {v:>7} {te:>7}')

Total segments test : 3678
Classes uniques     : 68

  ID                         Action   Train     Val    Test
------------------------------------------------------------
   0               take_short_brace      63      13      58
   1                  align_objects     212     111     246
   2                 take_pin_short     101      44     100
   3                 plug_short_pin     104      50      97
   4              take_tooth_washer     162      78     155
   5                       take_nut     169      93     175
   6                    tighten_nut     178      86     200
   7              check_instruction     404     357     539
   8             take_partial_model     140      55     107
   9                take_long_brace      47      26      46
  10                 take_screw_pin      31      14      28
  11               take_instruction       2       1       0
  12                put_instruction       3       1       0
  13                  take_pin_long      25   

In [5]:
# new code latest version
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

# ── Mapping 3 classes ─────────────────────────────────────
SEMANTIC_GROUPS_3 = {
    # 0 - MANIPULATION (take + put + pull)
    0:0, 2:0, 4:0, 5:0, 8:0, 9:0, 10:0, 11:0, 13:0,
    15:0, 17:0, 18:0, 20:0, 21:0, 23:0, 27:0, 43:0, 65:0,
    12:0, 14:0, 24:0, 35:0, 36:0, 40:0, 42:0, 44:0, 45:0,
    47:0, 48:0, 52:0, 54:0, 57:0, 59:0, 60:0, 62:0, 67:0,
    38:0, 41:0, 46:0, 49:0, 58:0, 61:0, 70:0, 72:0,
    # 1 - ASSEMBLAGE (fit + plug + tighten + loosen + align)
    1:1, 3:1, 6:1, 16:1, 19:1, 22:1, 25:1, 26:1, 28:1,
    30:1, 31:1, 32:1, 33:1, 34:1, 39:1, 50:1, 53:1,
    55:1, 56:1, 63:1, 64:1, 68:1, 69:1, 71:1, 73:1, 74:1,
    # 2 - VERIFICATION (check + browse)
    7:2, 29:2, 51:2,
}

NUM_CLASSES_3 = 3
CLASS_NAMES_3 = {0: 'MANIPULATION', 1: 'ASSEMBLAGE', 2: 'VERIFICATION'}

# Appliquer sur train + test + val
for df in [train_df, val_df, test_df]:
    df['label3'] = df['action_id'].map(SEMANTIC_GROUPS_3)

# Vérifier couverture
train_nan = train_df['label3'].isna().sum()
val_nan   = val_df['label3'].isna().sum()
test_nan  = test_df['label3'].isna().sum()
print(f'NaN train : {train_nan} | val : {val_nan} | test : {test_nan}')

# Distribution
print('\n=== DISTRIBUTION 3 CLASSES ===')
for split, df in [('TRAIN', train_df), ('VAL', val_df), ('TEST', test_df)]:
    counts = df['label3'].value_counts().sort_index()
    total  = counts.sum()
    print(f'\n{split} :')
    for cls, cnt in counts.items():
        print(f'  {CLASS_NAMES_3[cls]:15s} : {cnt:5d} ({cnt/total:.1%})')

# ── Class weights sur train+test combinés ────────────────
train_test_labels = pd.concat([
    train_df['label3'].dropna(),
    test_df['label3'].dropna()
]).astype(int).values

classes = np.array([0, 1, 2])
weights = compute_class_weight('balanced', classes=classes, y=train_test_labels)
weights = np.clip(weights, 0, 5.0)  # cap à 5 pour 3 classes
class_weights_tensor_3 = torch.tensor(weights, dtype=torch.float32)

print(f'\n=== CLASS WEIGHTS (cap=5) ===')
for i, w in enumerate(weights):
    print(f'  {CLASS_NAMES_3[i]:15s} : {w:.3f}')
print(f'\n✅ Mapping 3 classes prêt')

NaN train : 0 | val : 4 | test : 2

=== DISTRIBUTION 3 CLASSES ===

TRAIN :
  MANIPULATION    :  1840 (50.2%)
  ASSEMBLAGE      :  1371 (37.4%)
  VERIFICATION    :   456 (12.4%)

VAL :
  MANIPULATION    :   870 (45.2%)
  ASSEMBLAGE      :   672 (34.9%)
  VERIFICATION    :   382 (19.9%)

TEST :
  MANIPULATION    :  1689 (45.9%)
  ASSEMBLAGE      :  1382 (37.6%)
  VERIFICATION    :   605 (16.5%)

=== CLASS WEIGHTS (cap=5) ===
  MANIPULATION    : 0.694
  ASSEMBLAGE      : 0.889
  VERIFICATION    : 2.307

✅ Mapping 3 classes prêt


In [7]:
#cellule latest version
import pandas as pd

# Combiner train + test labels
train_test_df = pd.concat([train_df, test_df], ignore_index=True)

print(f'Train seul  : {len(train_df)} segments | {train_df["recording"].nunique()} recordings')
print(f'Test seul   : {len(test_df)} segments  | {test_df["recording"].nunique()} recordings')
print(f'Train+Test  : {len(train_test_df)} segments | {train_test_df["recording"].nunique()} recordings')

# Distribution combinée
counts = train_test_df['label3'].value_counts().sort_index()
total  = counts.sum()
print(f'\n=== DISTRIBUTION TRAIN+TEST ===')
for cls, cnt in counts.items():
    print(f'  {CLASS_NAMES_3[cls]:15s} : {cnt:5d} ({cnt/total:.1%})')

Train seul  : 3667 segments | 36 recordings
Test seul   : 3678 segments  | 32 recordings
Train+Test  : 7345 segments | 68 recordings

=== DISTRIBUTION TRAIN+TEST ===
  MANIPULATION    :  3529 (48.1%)
  ASSEMBLAGE      :  2753 (37.5%)
  VERIFICATION    :  1061 (14.4%)


In [ ]:
#bigru last version


In [2]:
with open(f'{PREP_DIR}scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open(f'{PREP_DIR}stride_map_train.pkl', 'rb') as f:
    stride_map_train = pickle.load(f)
with open(f'{PREP_DIR}stride_map_val.pkl', 'rb') as f:
    stride_map_val = pickle.load(f)
with open(f'{PREP_DIR}rec_to_file_train.pkl', 'rb') as f:
    rec_to_file_train = pickle.load(f)
with open(f'{PREP_DIR}rec_to_file_val.pkl', 'rb') as f:
    rec_to_file_val = pickle.load(f)

os.system('curl -L -o /kaggle/working/labels.zip "https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/778d7d5f-6631-4cb6-9451-f88c574c7991"')
os.system('unzip -q /kaggle/working/labels.zip -d /kaggle/working/labels/')

col_names = ['recording', 'action_id', 'action_name', 'frame_start', 'frame_end']
train_df  = pd.read_csv('/kaggle/working/labels/train.csv', header=None, names=col_names)
val_df    = pd.read_csv('/kaggle/working/labels/val.csv',   header=None, names=col_names)
train_df['frame_start'] = train_df['frame_start'].str.replace('.jpg','').astype(int)
train_df['frame_end']   = train_df['frame_end'].str.replace('.jpg','').astype(int)
val_df['frame_start']   = val_df['frame_start'].str.replace('.jpg','').astype(int)
val_df['frame_end']     = val_df['frame_end'].str.replace('.jpg','').astype(int)
print(f'Train : {len(train_df)} segments | Val : {len(val_df)} segments ✅')

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

Train : 3667 segments | Val : 1928 segments ✅


100 67816  100 67816    0     0  74562      0 --:--:-- --:--:-- --:--:-- 74523


In [3]:
SEMANTIC_GROUPS_16 = {
    0:0, 9:0, 15:0,
    2:1, 10:1, 13:1, 20:1, 65:1,
    4:2, 5:2, 17:2, 18:2,
    8:3, 21:3, 23:3, 27:3, 43:3,
    14:4, 35:4, 42:4, 47:4, 67:4,
    40:5, 54:5, 57:5, 62:5,
    24:6, 36:6, 44:6, 45:6, 48:6, 52:6, 59:6, 60:6,
    3:7, 16:7, 22:7, 25:7, 26:7, 28:7, 71:7,
    6:8, 19:8, 31:8, 32:8, 34:8, 68:8, 73:8,
    30:9, 33:9, 50:9, 53:9, 55:9, 56:9, 64:9, 69:9,
    38:10, 41:10, 46:10, 49:10, 58:10, 61:10, 70:10, 72:10,
    39:11, 63:11, 74:11,
    7:12,
    1:13,
    11:14, 12:14, 29:14, 51:14,
}
NUM_CLASSES = len(set(SEMANTIC_GROUPS_16.values()))

all_classes  = sorted(set(SEMANTIC_GROUPS_16.values()))
class_to_idx = {c: c for c in all_classes}  # déjà 0..15
idx_to_class = {c: c for c in all_classes}

train_df['label'] = train_df['action_id'].map(SEMANTIC_GROUPS_16)
val_df['label']   = val_df['action_id'].map(SEMANTIC_GROUPS_16)

# Class weights
y_train_all = train_df['label'].dropna().astype(int).values
classes     = np.array(sorted(np.unique(y_train_all)))
weights     = compute_class_weight('balanced', classes=classes, y=y_train_all)
weights     = np.clip(weights, 0, 10.0)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

print(f'NUM_CLASSES : {NUM_CLASSES}')
print(f'Class weights | min={weights.min():.3f} | max={weights.max():.3f}')
print('✅ Mapping 16 classes prêt')

NUM_CLASSES : 15
Class weights | min=0.373 | max=4.527
✅ Mapping 16 classes prêt


In [6]:
import os, re, random, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

# ── Chemins ───────────────────────────────────────────────
TRAIN_FEAT_DIR = '/kaggle/input/datasets/hibabou/industrial/data/train/'
VAL_FEAT_DIR   = '/kaggle/input/datasets/hibabou/industrial/data/val/'
TEST_FEAT_DIR  = '/kaggle/input/datasets/hibabou/featuressupp/sf_features_test/'
PREP_DIR       = '/kaggle/input/datasets/hibabou/prepbigru/kaggle/working/'
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FEAT_DIM       = 2304
NUM_CLASSES    = 3

# ── Semantic groups ───────────────────────────────────────
SEMANTIC_GROUPS_3 = {
    0:0, 2:0, 4:0, 5:0, 8:0, 9:0, 10:0, 11:0, 13:0,
    15:0, 17:0, 18:0, 20:0, 21:0, 23:0, 27:0, 43:0, 65:0,
    12:0, 14:0, 24:0, 35:0, 36:0, 40:0, 42:0, 44:0, 45:0,
    47:0, 48:0, 52:0, 54:0, 57:0, 59:0, 60:0, 62:0, 67:0,
    38:0, 41:0, 46:0, 49:0, 58:0, 61:0, 70:0, 72:0,
    1:1, 3:1, 6:1, 16:1, 19:1, 22:1, 25:1, 26:1, 28:1,
    30:1, 31:1, 32:1, 33:1, 34:1, 39:1, 50:1, 53:1,
    55:1, 56:1, 63:1, 64:1, 68:1, 69:1, 71:1, 73:1, 74:1,
    7:2, 29:2, 51:2,
}
CLASS_NAMES_3 = {0: 'MANIPULATION', 1: 'ASSEMBLAGE', 2: 'VERIFICATION'}

# ── Labels CSV ────────────────────────────────────────────
os.system('curl -L -o /kaggle/working/labels.zip "https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/778d7d5f-6631-4cb6-9451-f88c574c7991"')
os.system('unzip -q /kaggle/working/labels.zip -d /kaggle/working/labels/')

col_names = ['recording', 'action_id', 'action_name', 'frame_start', 'frame_end']
train_df  = pd.read_csv('/kaggle/working/labels/train.csv', header=None, names=col_names)
val_df    = pd.read_csv('/kaggle/working/labels/val.csv',   header=None, names=col_names)
test_df   = pd.read_csv('/kaggle/working/labels/test.csv',  header=None, names=col_names)

for df in [train_df, val_df, test_df]:
    df['frame_start'] = df['frame_start'].str.replace('.jpg','').astype(int)
    df['frame_end']   = df['frame_end'].str.replace('.jpg','').astype(int)
    df['label3']      = df['action_id'].map(SEMANTIC_GROUPS_3)

print(f'Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)} ✅')

# ── Charger pkl ───────────────────────────────────────────
with open(f'{PREP_DIR}scaler.pkl',            'rb') as f: scaler            = pickle.load(f)
with open(f'{PREP_DIR}stride_map_train.pkl',  'rb') as f: stride_map_train  = pickle.load(f)
with open(f'{PREP_DIR}stride_map_val.pkl',    'rb') as f: stride_map_val    = pickle.load(f)
with open(f'{PREP_DIR}rec_to_file_train.pkl', 'rb') as f: rec_to_file_train = pickle.load(f)
with open(f'{PREP_DIR}rec_to_file_val.pkl',   'rb') as f: rec_to_file_val   = pickle.load(f)

# ── Stride map test ───────────────────────────────────────
rec_to_file_test = {}
for f in os.listdir(TEST_FEAT_DIR):
    if f.endswith('.npy'):
        rec = re.sub(r'^test_p\d+_', '', f).replace('_features.npy', '')
        rec_to_file_test[rec] = os.path.join(TEST_FEAT_DIR, f)

stride_map_test = {}
for rec, path in rec_to_file_test.items():
    sub = test_df[test_df['recording'] == rec]
    if len(sub) == 0:
        continue
    arr    = np.load(path)
    f_min  = sub['frame_start'].min()
    f_max  = sub['frame_end'].max()
    n_feat = arr.shape[0]
    total  = f_max - f_min
    stride = round((total - 32) / (n_feat - 1)) if n_feat > 1 else 1
    stride = max(1, stride)
    stride_map_test[rec] = {'stride': stride, 'f_min': f_min, 'n_feats': n_feat}

# ── Class weights ─────────────────────────────────────────
train_test_labels = pd.concat([
    train_df['label3'].dropna(),
    test_df['label3'].dropna()
]).astype(int).values

classes = np.array([0, 1, 2])
weights = compute_class_weight('balanced', classes=classes, y=train_test_labels)
weights = np.clip(weights, 0, 5.0)
class_weights_tensor_3 = torch.tensor(weights, dtype=torch.float32)

# ── Fonctions ─────────────────────────────────────────────
def get_feat_indices(frame_start, frame_end, f_min, n_feats, stride):
    i0 = max(0, round((frame_start - f_min) / stride))
    i1 = min(n_feats, round((frame_end   - f_min) / stride))
    return i0, i1

def compute_f1_metrics(all_preds, all_labels, num_classes):
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    mask       = all_labels >= 0
    f1_macro    = f1_score(all_labels[mask], all_preds[mask], average='macro',    zero_division=0)
    f1_weighted = f1_score(all_labels[mask], all_preds[mask], average='weighted', zero_division=0)
    f1_per      = f1_score(all_labels[mask], all_preds[mask], average=None,
                           zero_division=0, labels=list(range(num_classes)))
    return f1_macro, f1_weighted, f1_per

print(f'Device : {DEVICE}')
print(f'Train : {len(stride_map_train)} | Val : {len(stride_map_val)} | Test : {len(stride_map_test)} ✅')
print(f'Class weights : {weights.round(3)}')

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 67816  100 67816    0     0   121k      0 --:--:-- --:--:-- --:--:--  121k
replace /kaggle/working/labels/test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename:  NULL
(EOF or read error, treating as "[N]one" ...)


Train : 3667 | Val : 1928 | Test : 3678 ✅
Device : cuda
Train : 36 | Val : 16 | Test : 32 ✅
Class weights : [0.694 0.889 2.307]


In [7]:
import os

os.system('curl -L -o /kaggle/working/labels.zip "https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/778d7d5f-6631-4cb6-9451-f88c574c7991"')
os.system('unzip -q /kaggle/working/labels.zip -d /kaggle/working/labels/')

import pandas as pd
col_names = ['recording', 'action_id', 'action_name', 'frame_start', 'frame_end']
test_df   = pd.read_csv('/kaggle/working/labels/test.csv',  header=None, names=col_names)
train_df  = pd.read_csv('/kaggle/working/labels/train.csv', header=None, names=col_names)
val_df    = pd.read_csv('/kaggle/working/labels/val.csv',   header=None, names=col_names)

for df in [train_df, val_df, test_df]:
    df['frame_start'] = df['frame_start'].str.replace('.jpg','').astype(int)
    df['frame_end']   = df['frame_end'].str.replace('.jpg','').astype(int)
    df['label3']      = df['action_id'].map(SEMANTIC_GROUPS_3)

print(f'Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)} ✅')

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

Train : 3667 | Val : 1928 | Test : 3678 ✅


100 67816  100 67816    0     0   121k      0 --:--:-- --:--:-- --:--:--  121k
replace /kaggle/working/labels/test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename:  NULL
(EOF or read error, treating as "[N]one" ...)


In [8]:
#v8
import os, re, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

# ── Chemins ───────────────────────────────────────────────
TRAIN_FEAT_DIR = '/kaggle/input/datasets/hibabou/industrial/data/train/'
VAL_FEAT_DIR   = '/kaggle/input/datasets/hibabou/industrial/data/val/'
TEST_FEAT_DIR  = '/kaggle/input/datasets/hibabou/featuressupp/sf_features_test/'
PREP_DIR       = '/kaggle/input/datasets/hibabou/prepbigru/kaggle/working/'
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FEAT_DIM       = 2304
NUM_CLASSES    = 3

# ── Charger pkl ───────────────────────────────────────────
with open(f'{PREP_DIR}scaler.pkl',            'rb') as f: scaler           = pickle.load(f)
with open(f'{PREP_DIR}stride_map_train.pkl',  'rb') as f: stride_map_train = pickle.load(f)
with open(f'{PREP_DIR}stride_map_val.pkl',    'rb') as f: stride_map_val   = pickle.load(f)
with open(f'{PREP_DIR}rec_to_file_train.pkl', 'rb') as f: rec_to_file_train = pickle.load(f)
with open(f'{PREP_DIR}rec_to_file_val.pkl',   'rb') as f: rec_to_file_val   = pickle.load(f)

print(f'Train : {len(stride_map_train)} | Val : {len(stride_map_val)} ✅')

# ── Stride map test ───────────────────────────────────────
rec_to_file_test = {}
for f in os.listdir(TEST_FEAT_DIR):
    if f.endswith('.npy'):
        rec = re.sub(r'^test_p\d+_', '', f).replace('_features.npy', '')
        rec_to_file_test[rec] = os.path.join(TEST_FEAT_DIR, f)

stride_map_test = {}
for rec, path in rec_to_file_test.items():
    sub = test_df[test_df['recording'] == rec]
    if len(sub) == 0:
        continue
    arr    = np.load(path)
    f_min  = sub['frame_start'].min()
    f_max  = sub['frame_end'].max()
    n_feat = arr.shape[0]
    total  = f_max - f_min
    stride = round((total - 32) / (n_feat - 1)) if n_feat > 1 else 1
    stride = max(1, stride)
    stride_map_test[rec] = {'stride': stride, 'f_min': f_min, 'n_feats': n_feat}

print(f'Test  : {len(stride_map_test)} recordings mappés ✅')

# ── Fonction mapping ──────────────────────────────────────
def get_feat_indices(frame_start, frame_end, f_min, n_feats, stride):
    i0 = max(0, round((frame_start - f_min) / stride))
    i1 = min(n_feats, round((frame_end   - f_min) / stride))
    return i0, i1

def compute_f1_metrics(all_preds, all_labels, num_classes):
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    mask       = all_labels >= 0
    preds      = all_preds[mask]
    labels     = all_labels[mask]
    f1_macro    = f1_score(labels, preds, average='macro',    zero_division=0)
    f1_weighted = f1_score(labels, preds, average='weighted', zero_division=0)
    f1_per      = f1_score(labels, preds, average=None,       zero_division=0,
                           labels=list(range(num_classes)))
    return f1_macro, f1_weighted, f1_per

print('✅ Fonctions utilitaires prêtes')

Train : 36 | Val : 16 ✅
Test  : 32 recordings mappés ✅
✅ Fonctions utilitaires prêtes


In [9]:
# v8── Dataset ───────────────────────────────────────────────
class IndustRealDataset(Dataset):
    def __init__(self, dfs, rec_to_files, stride_maps, scaler,
                 semantic_groups, clip_val=5.0, seq_len=128,
                 step=64, is_train=False,
                 oversample_cls=2, oversample_factor=2,
                 undersample_cls=0, undersample_ratio=0.7):
        self.samples  = []
        self.is_train = is_train

        # Accepte liste de (df, rec_to_file, stride_map)
        if not isinstance(dfs, list):
            dfs          = [dfs]
            rec_to_files = [rec_to_files]
            stride_maps  = [stride_maps]

        for df, rec_to_file, stride_map in zip(dfs, rec_to_files, stride_maps):
            for rec, info in stride_map.items():
                sub = df[df['recording'] == rec]
                if len(sub) == 0:
                    continue

                arr = np.load(rec_to_file[rec]).astype(np.float32)
                arr = scaler.transform(arr).astype(np.float32)
                arr = np.clip(arr, -5.0, 5.0)

                labels = np.full(arr.shape[0], -1, dtype=np.int64)
                for _, row in sub.iterrows():
                    if row['action_id'] not in semantic_groups:
                        continue
                    i0, i1 = get_feat_indices(
                        row['frame_start'], row['frame_end'],
                        info['f_min'], info['n_feats'], info['stride']
                    )
                    labels[i0:i1] = semantic_groups[row['action_id']]

                for start in range(0, len(arr) - seq_len + 1, step):
                    end          = start + seq_len
                    chunk_feats  = arr[start:end]
                    chunk_labels = labels[start:end]
                    labeled      = chunk_labels[chunk_labels >= 0]

                    if len(labeled) < seq_len // 4:
                        continue

                    dominant = np.bincount(labeled, minlength=NUM_CLASSES).argmax()

                    # Undersampling MANIPULATION
                    if is_train and dominant == undersample_cls:
                        if np.random.random() > undersample_ratio:
                            continue

                    self.samples.append({
                        'features' : torch.tensor(chunk_feats,  dtype=torch.float32),
                        'labels'   : torch.tensor(chunk_labels, dtype=torch.long),
                        'dominant' : dominant
                    })

        # Oversampling VERIFICATION
        if is_train:
            verif_samples = [s for s in self.samples if s['dominant'] == oversample_cls]
            for _ in range(oversample_factor - 1):
                self.samples.extend(verif_samples)
            random.shuffle(self.samples)

        # Stats
        counts = {0:0, 1:0, 2:0}
        for s in self.samples:
            counts[s['dominant']] += 1
        print(f'Dataset : {len(self.samples)} segments | train={is_train}')
        for cls, cnt in counts.items():
            print(f'  {CLASS_NAMES_3[cls]:15s} : {cnt} ({cnt/len(self.samples):.1%})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        feats  = self.samples[idx]['features'].clone()
        labels = self.samples[idx]['labels'].clone()

        if self.is_train:
            # Flip temporel
            if random.random() > 0.5:
                feats  = feats.flip(0)
                labels = labels.flip(0)
            # Bruit gaussien
            if random.random() > 0.5:
                feats = feats + torch.randn_like(feats) * 0.05
            # Time masking
            if random.random() > 0.7:
                mask_len   = random.randint(5, 20)
                mask_start = random.randint(0, max(0, len(feats) - mask_len))
                feats[mask_start:mask_start+mask_len] = 0.0

        return feats, labels

def collate_fn(batch):
    feats, labels = zip(*batch)
    return torch.stack(feats), torch.stack(labels), \
           torch.tensor([f.shape[0] for f in feats])

# Instancier train = train + test
SEQ_LEN    = 128
STEP       = 64
BATCH_SIZE = 16

train_dataset = IndustRealDataset(
    dfs          = [train_df,          test_df],
    rec_to_files = [rec_to_file_train, rec_to_file_test],
    stride_maps  = [stride_map_train,  stride_map_test],
    scaler       = scaler,
    semantic_groups = SEMANTIC_GROUPS_3,
    seq_len      = SEQ_LEN,
    step         = STEP,
    is_train     = True,
    oversample_cls    = 2,   # VERIFICATION
    oversample_factor = 3,   # ×3
    undersample_cls   = 0,   # MANIPULATION
    undersample_ratio = 0.7  # garder 70%
)

val_dataset = IndustRealDataset(
    dfs          = val_df,
    rec_to_files = rec_to_file_val,
    stride_maps  = stride_map_val,
    scaler       = scaler,
    semantic_groups = SEMANTIC_GROUPS_3,
    seq_len      = SEQ_LEN,
    step         = STEP,
    is_train     = False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)

print(f'\nTrain : {len(train_loader)} batches')
print(f'Val   : {len(val_loader)} batches')
print(f'Device : {DEVICE}')
print('✅ DataLoader prêt')

Dataset : 2486 segments | train=True
  MANIPULATION    : 432 (17.4%)
  ASSEMBLAGE      : 1208 (48.6%)
  VERIFICATION    : 846 (34.0%)
Dataset : 270 segments | train=False
  MANIPULATION    : 62 (23.0%)
  ASSEMBLAGE      : 153 (56.7%)
  VERIFICATION    : 55 (20.4%)

Train : 156 batches
Val   : 17 batches
Device : cuda
✅ DataLoader prêt


In [12]:
# Vérifier distribution réelle avant oversampling
print('Distribution dominante par chunk (avant sampling) :')
from collections import Counter

temp_counts = Counter()
for s in train_dataset.samples:
    temp_counts[s['dominant']] += 1

total = sum(temp_counts.values())
for cls, cnt in sorted(temp_counts.items()):
    print(f'  {CLASS_NAMES_3[cls]:15s} : {cnt} ({cnt/total:.1%})')

Distribution dominante par chunk (avant sampling) :
  MANIPULATION    : 432 (17.4%)
  ASSEMBLAGE      : 1208 (48.6%)
  VERIFICATION    : 846 (34.0%)


In [14]:
train_dataset = IndustRealDataset(
    dfs          = [train_df,          test_df],
    rec_to_files = [rec_to_file_train, rec_to_file_test],
    stride_maps  = [stride_map_train,  stride_map_test],
    scaler       = scaler,
    semantic_groups = SEMANTIC_GROUPS_3,
    seq_len      = SEQ_LEN,
    step         = STEP,
    is_train     = True,
    oversample_cls    = 2,   # VERIFICATION ×2
    oversample_factor = 2,
    undersample_cls   = 1,   # ASSEMBLAGE
    undersample_ratio = 0.35 # garder 35%
)
val_dataset = IndustRealDataset(
    dfs          = val_df,
    rec_to_files = rec_to_file_val,
    stride_maps  = stride_map_val,
    scaler       = scaler,
    semantic_groups = SEMANTIC_GROUPS_3,
    seq_len      = SEQ_LEN,
    step         = STEP,
    is_train     = False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)

print(f'Train : {len(train_loader)} batches')
print(f'Val   : {len(val_loader)} batches')

# Vérifier distribution
from collections import Counter
counts = Counter(s['dominant'] for s in train_dataset.samples)
total  = sum(counts.values())
print(f'\nDistribution train :')
for cls in range(NUM_CLASSES):
    cnt = counts.get(cls, 0)
    print(f'  {CLASS_NAMES_3[cls]:15s} : {cnt} ({cnt/total:.1%})')

Dataset : 1584 segments | train=True
  MANIPULATION    : 601 (37.9%)
  ASSEMBLAGE      : 419 (26.5%)
  VERIFICATION    : 564 (35.6%)
Dataset : 270 segments | train=False
  MANIPULATION    : 62 (23.0%)
  ASSEMBLAGE      : 153 (56.7%)
  VERIFICATION    : 55 (20.4%)
Train : 99 batches
Val   : 17 batches

Distribution train :
  MANIPULATION    : 601 (37.9%)
  ASSEMBLAGE      : 419 (26.5%)
  VERIFICATION    : 564 (35.6%)


In [15]:
#v8
# ── Modèle BiGRU ──────────────────────────────────────────
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, gru_out, lengths=None):
        scores = self.attn(gru_out).squeeze(-1)
        if lengths is not None:
            for i, l in enumerate(lengths):
                if l < scores.shape[1]:
                    scores[i, l:] = float('-inf')
        return torch.softmax(scores, dim=1)


class BiGRUDualHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers,
                 num_classes, dropout=0.4):
        super().__init__()

        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # BiGRU
        self.bigru = nn.GRU(
            input_size    = hidden_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if num_layers > 1 else 0.0
        )

        self.norm      = nn.LayerNorm(hidden_dim * 2)
        self.dropout   = nn.Dropout(dropout)
        self.attention = TemporalAttention(hidden_dim)

        # Action head
        self.action_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, num_classes)
        )

        # Anomaly head
        self.anomaly_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x, lengths=None):
        B, T, _ = x.shape

        x = self.input_proj(x)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            gru_out, _ = self.bigru(packed)
            gru_out, _ = nn.utils.rnn.pad_packed_sequence(
                gru_out, batch_first=True, total_length=T)
        else:
            gru_out, _ = self.bigru(x)

        gru_out      = self.norm(self.dropout(gru_out))
        attn_weights = self.attention(gru_out, lengths)
        action_logits = self.action_head(gru_out)
        anomaly_score = self.anomaly_head(gru_out)

        return action_logits, anomaly_score, attn_weights, gru_out

# Instancier
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT    = 0.4

model = BiGRUDualHead(
    input_dim   = FEAT_DIM,
    hidden_dim  = HIDDEN_DIM,
    num_layers  = NUM_LAYERS,
    num_classes = NUM_CLASSES,
    dropout     = DROPOUT
).to(DEVICE)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'BiGRU 3 classes')
print(f'  Hidden    : {HIDDEN_DIM}x2 bidirectionnel')
print(f'  Layers    : {NUM_LAYERS}')
print(f'  Classes   : {NUM_CLASSES}')
print(f'  Params    : {params:,}')
print(f'  Device    : {DEVICE}')
print('✅ Modèle prêt')

BiGRU 3 classes
  Hidden    : 512x2 bidirectionnel
  Layers    : 2
  Classes   : 3
  Params    : 10,504,709
  Device    : cuda
✅ Modèle prêt


In [16]:
#v8
# ── Training functions ────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for feats, labels, lengths in loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()

        action_logits, _, _, _ = model(feats, lengths)
        B, T, C = action_logits.shape
        loss = criterion(action_logits.reshape(B*T, C), labels.reshape(B*T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = action_logits.argmax(dim=-1).reshape(B*T)
        labs  = labels.reshape(B*T)
        mask  = labs >= 0
        all_preds.extend(preds[mask].cpu().numpy())
        all_labels.extend(labs[mask].cpu().numpy())

    f1_macro, f1_weighted, _ = compute_f1_metrics(all_preds, all_labels, NUM_CLASSES)
    return total_loss / len(loader), f1_macro, f1_weighted


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for feats, labels, lengths in loader:
            feats, labels = feats.to(device), labels.to(device)
            action_logits, _, _, _ = model(feats, lengths)
            B, T, C = action_logits.shape
            loss = criterion(action_logits.reshape(B*T, C), labels.reshape(B*T))
            total_loss += loss.item()
            preds = action_logits.argmax(dim=-1).reshape(B*T)
            labs  = labels.reshape(B*T)
            mask  = labs >= 0
            all_preds.extend(preds[mask].cpu().numpy())
            all_labels.extend(labs[mask].cpu().numpy())

    f1_macro, f1_weighted, f1_per = compute_f1_metrics(
        all_preds, all_labels, NUM_CLASSES)
    return total_loss / len(loader), f1_macro, f1_weighted, f1_per


# ── Loss + Optimizer ──────────────────────────────────────
criterion = nn.CrossEntropyLoss(
    weight        = class_weights_tensor_3.to(DEVICE),
    ignore_index  = -1,
    label_smoothing = 0.1
)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

# ── Training loop ─────────────────────────────────────────
NUM_EPOCHS   = 60
PATIENCE     = 12
best_f1      = 0.0
patience_ctr = 0

print(f'Training BiGRU 3 classes | train+test | {DEVICE}')
print(f'{"Epoch":>5} {"TLoss":>8} {"TF1":>7} {"VLoss":>8} {"VF1":>7} '
      f'{"MANIP":>7} {"ASSEM":>7} {"VERIF":>7}')
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_f1, _        = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_f1, _, val_f1_pc = val_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step(val_f1)

    manip = val_f1_pc[0] if len(val_f1_pc) > 0 else 0
    assem = val_f1_pc[1] if len(val_f1_pc) > 1 else 0
    verif = val_f1_pc[2] if len(val_f1_pc) > 2 else 0

    print(f'{epoch:>5} {train_loss:>8.4f} {train_f1:>7.4f} '
          f'{val_loss:>8.4f} {val_f1:>7.4f} '
          f'{manip:>7.4f} {assem:>7.4f} {verif:>7.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({
            'epoch'          : epoch,
            'model_state'    : model.state_dict(),
            'optimizer'      : optimizer.state_dict(),
            'val_f1'         : val_f1,
            'val_f1_per_class': val_f1_pc.tolist(),
            'semantic_groups': SEMANTIC_GROUPS_3,
            'num_classes'    : NUM_CLASSES,
            'hidden_dim'     : HIDDEN_DIM,
            'class_names'    : CLASS_NAMES_3,
        }, '/kaggle/working/bigru_3classes_best.pth')
        print(f'       ✅ Saved (val_f1={val_f1:.4f} | '
              f'MANIP={manip:.3f} ASSEM={assem:.3f} VERIF={verif:.3f})')

    patience_ctr = 0 if val_f1 > best_f1 - 0.001 else patience_ctr + 1
    if patience_ctr >= PATIENCE:
        print(f'\nEarly stopping epoch {epoch}')
        break

print(f'\nMeilleur val F1 macro : {best_f1:.4f}')
print(f'  MANIPULATION : {best_f1:.4f}')

Training BiGRU 3 classes | train+test | cuda
Epoch    TLoss     TF1    VLoss     VF1   MANIP   ASSEM   VERIF
-----------------------------------------------------------------
    1   1.0098  0.1761   0.9930  0.1580  0.0601  0.0008  0.4130
       ✅ Saved (val_f1=0.1580 | MANIP=0.060 ASSEM=0.001 VERIF=0.413)
    2   0.9725  0.2754   1.0533  0.3979  0.3341  0.4679  0.3916
       ✅ Saved (val_f1=0.3979 | MANIP=0.334 ASSEM=0.468 VERIF=0.392)
    3   0.9444  0.3602   1.0388  0.3001  0.2155  0.2616  0.4233
    4   0.8879  0.4507   1.0187  0.4085  0.1307  0.6423  0.4526
       ✅ Saved (val_f1=0.4085 | MANIP=0.131 ASSEM=0.642 VERIF=0.453)
    5   0.8431  0.5040   1.0878  0.4498  0.3438  0.5906  0.4151
       ✅ Saved (val_f1=0.4498 | MANIP=0.344 ASSEM=0.591 VERIF=0.415)
    6   0.7946  0.5550   1.0647  0.4329  0.3106  0.5735  0.4146
    7   0.7675  0.5850   1.1379  0.4132  0.2650  0.6038  0.3707
    8   0.7376  0.6120   1.1984  0.3996  0.2722  0.6158  0.3108
    9   0.7110  0.6396   1.2038  0.44

In [17]:
import os

# Vérifier le checkpoint
print(os.path.exists('/kaggle/working/bigru_3classes_best.pth'))
print(f'Taille : {os.path.getsize("/kaggle/working/bigru_3classes_best.pth")/1e6:.1f} MB')

# Zipper
os.system('zip /kaggle/working/checkpoint_3classes.zip /kaggle/working/bigru_3classes_best.pth')
os.system('ls -lh /kaggle/working/checkpoint_3classes.zip')

True
Taille : 119.8 MB
  adding: kaggle/working/bigru_3classes_best.pth (deflated 8%)
-rw-r--r-- 1 root root 106M May 19 11:08 /kaggle/working/checkpoint_3classes.zip


0

In [4]:
def get_feat_indices(frame_start, frame_end, f_min, n_feats, stride):
    i0 = max(0, round((frame_start - f_min) / stride))
    i1 = min(n_feats, round((frame_end   - f_min) / stride))
    return i0, i1

def compute_f1(all_preds, all_labels, num_classes):
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    mask       = all_labels >= 0
    preds      = all_preds[mask]
    labels     = all_labels[mask]
    f1_macro    = f1_score(labels, preds, average='macro',    zero_division=0)
    f1_weighted = f1_score(labels, preds, average='weighted', zero_division=0)
    f1_per_class = f1_score(labels, preds, average=None,      zero_division=0,
                            labels=list(range(num_classes)))
    return f1_macro, f1_weighted, f1_per_class

print('✅ Fonctions utilitaires prêtes')

✅ Fonctions utilitaires prêtes


In [5]:
class IndustRealDataset(Dataset):
    def __init__(self, df, rec_to_file, stride_map, scaler,
                 semantic_groups, clip_val=5.0, seq_len=128, 
                 step=64, is_train=False):
        self.samples  = []
        self.is_train = is_train

        for rec, info in stride_map.items():
            sub = df[df['recording'] == rec]
            if len(sub) == 0:
                continue
            arr = np.load(rec_to_file[rec]).astype(np.float32)
            arr = scaler.transform(arr).astype(np.float32)
            arr = np.clip(arr, -clip_val, clip_val)

            labels = np.full(arr.shape[0], -1, dtype=np.int64)
            for _, row in sub.iterrows():
                if row['action_id'] not in semantic_groups:
                    continue
                i0, i1 = get_feat_indices(
                    row['frame_start'], row['frame_end'],
                    info['f_min'], info['n_feats'], info['stride']
                )
                labels[i0:i1] = semantic_groups[row['action_id']]

            for start in range(0, len(arr) - seq_len + 1, step):
                end          = start + seq_len
                chunk_feats  = arr[start:end]
                chunk_labels = labels[start:end]
                if (chunk_labels >= 0).sum() < seq_len // 4:
                    continue
                self.samples.append({
                    'features': torch.tensor(chunk_feats,  dtype=torch.float32),
                    'labels'  : torch.tensor(chunk_labels, dtype=torch.long),
                })

        print(f'Dataset : {len(self.samples)} segments | seq_len={seq_len} | train={is_train}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        feats  = self.samples[idx]['features'].clone()
        labels = self.samples[idx]['labels'].clone()
        if self.is_train:
            if random.random() > 0.5:
                feats  = feats.flip(0)
                labels = labels.flip(0)
            if random.random() > 0.5:
                feats = feats + torch.randn_like(feats) * 0.05
        return feats, labels

def collate_fn(batch):
    feats, labels = zip(*batch)
    feats_padded  = torch.stack(feats)
    labels_padded = torch.stack(labels)
    lengths = torch.tensor([f.shape[0] for f in feats])
    return feats_padded, labels_padded, lengths

SEQ_LEN    = 128
STEP       = 64
BATCH_SIZE = 16

train_dataset = IndustRealDataset(
    train_df, rec_to_file_train, stride_map_train,
    scaler, SEMANTIC_GROUPS_16, seq_len=SEQ_LEN, 
    step=STEP, is_train=True
)
val_dataset = IndustRealDataset(
    val_df, rec_to_file_val, stride_map_val,
    scaler, SEMANTIC_GROUPS_16, seq_len=SEQ_LEN,
    step=STEP, is_train=False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)

print(f'Train : {len(train_dataset)} segments | {len(train_loader)} batches')
print(f'Val   : {len(val_dataset)} segments   | {len(val_loader)} batches')
print('✅ DataLoader prêt')

Dataset : 745 segments | seq_len=128 | train=True
Dataset : 270 segments | seq_len=128 | train=False
Train : 745 segments | 47 batches
Val   : 270 segments   | 17 batches
✅ DataLoader prêt


In [6]:
# Test fiabilité extraction : même classe = features similaires ?
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print('=== TEST FIABILITÉ EXTRACTION ===\n')

# Pour chaque classe groupée, calculer similarité intra-classe vs inter-classe
X_by_class = {}
for rec, info in stride_map_train.items():
    sub = train_df[train_df['recording'] == rec]
    if len(sub) == 0:
        continue
    arr = np.load(rec_to_file_train[rec]).astype(np.float32)
    arr = scaler.transform(arr)
    arr = np.clip(arr, -5.0, 5.0)
    
    labels = np.full(arr.shape[0], -1, dtype=np.int64)
    for _, row in sub.iterrows():
        if row['action_id'] not in SEMANTIC_GROUPS_16:
            continue
        i0, i1 = get_feat_indices(
            row['frame_start'], row['frame_end'],
            info['f_min'], info['n_feats'], info['stride']
        )
        labels[i0:i1] = SEMANTIC_GROUPS_16[row['action_id']]
    
    for cls in range(NUM_CLASSES):
        mask = labels == cls
        if mask.sum() > 0:
            if cls not in X_by_class:
                X_by_class[cls] = []
            # Prendre max 50 frames par recording
            idx = np.where(mask)[0][:50]
            X_by_class[cls].append(arr[idx])

# Calculer similarité intra vs inter
intra_sims = []
inter_sims = []

CLASS_NAMES = {
    0:'take_brace', 1:'take_pin', 2:'take_nut',
    3:'take_wheel', 4:'put_pin', 5:'put_nut',
    6:'put_wheel', 7:'fit_pin', 8:'fit_nut',
    9:'fit_wheel', 10:'pull', 11:'loosen',
    12:'tighten', 13:'align', 14:'check/browse',
    15:'objects'
}

print(f'{"Classe":>4} {"Nom":>15} {"Intra sim":>10} {"Frames":>8}')
print('-' * 45)
for cls in range(NUM_CLASSES):
    if cls not in X_by_class:
        continue
    X_cls = np.vstack(X_by_class[cls])
    # Subsample
    idx = np.random.choice(len(X_cls), min(200, len(X_cls)), replace=False)
    X_s = X_cls[idx]
    
    # Intra-classe similarity
    sim = cosine_similarity(X_s)
    np.fill_diagonal(sim, 0)
    intra = sim.mean()
    intra_sims.append(intra)
    
    print(f'{cls:>4} {CLASS_NAMES[cls]:>15} {intra:>10.4f} {len(X_cls):>8}')

# Inter-classe (random 2 classes)
all_X = []
all_y = []
for cls, chunks in X_by_class.items():
    X_c = np.vstack(chunks)
    idx = np.random.choice(len(X_c), min(50, len(X_c)), replace=False)
    all_X.append(X_c[idx])
    all_y.extend([cls] * len(idx))

all_X = np.vstack(all_X)
all_y = np.array(all_y)
sim_matrix = cosine_similarity(all_X)

for i in range(len(all_y)):
    for j in range(i+1, min(i+10, len(all_y))):
        if all_y[i] != all_y[j]:
            inter_sims.append(sim_matrix[i,j])

print(f'\n=== RÉSUMÉ ===')
print(f'Similarité intra-classe moyenne : {np.mean(intra_sims):.4f}')
print(f'Similarité inter-classe moyenne : {np.mean(inter_sims):.4f}')
print(f'Ratio intra/inter               : {np.mean(intra_sims)/np.mean(inter_sims):.3f}')
print()
if np.mean(intra_sims) > np.mean(inter_sims):
    print('✅ EXTRACTION FIABLE — classes distinctes dans espace features')
else:
    print('⚠️  Classes très similaires visuellement — problème dataset pas extraction')

=== TEST FIABILITÉ EXTRACTION ===

Classe             Nom  Intra sim   Frames
---------------------------------------------
   0      take_brace     0.0310      858
   1        take_pin     0.0197     1269
   2        take_nut     0.0170     1728
   3      take_wheel     0.0119     1760
   4         put_pin     0.0464      338
   5         put_nut     0.0431      476
   6       put_wheel     0.0292     1538
   7         fit_pin     0.0379     1693
   8         fit_nut     0.0488     1800
   9       fit_wheel     0.0162     1412
  10            pull     0.0330      680
  11          loosen     0.0375      767
  12         tighten     0.0336     1497
  13           align     0.0283     1381
  14    check/browse     0.0482      849

=== RÉSUMÉ ===
Similarité intra-classe moyenne : 0.0321
Similarité inter-classe moyenne : 0.0312
Ratio intra/inter               : 1.030

✅ EXTRACTION FIABLE — classes distinctes dans espace features


In [7]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, gru_out, lengths=None):
        scores = self.attn(gru_out).squeeze(-1)
        if lengths is not None:
            for i, l in enumerate(lengths):
                scores[i, l:] = float('-inf')
        return torch.softmax(scores, dim=1)

class BiGRUDualHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers,
                 num_classes, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.bigru = nn.GRU(
            input_size=hidden_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.attention = TemporalAttention(hidden_dim)
        self.dropout   = nn.Dropout(dropout)
        self.norm      = nn.LayerNorm(hidden_dim * 2)
        self.action_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        self.anomaly_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x, lengths=None):
        B, T, _ = x.shape
        x = self.input_proj(x)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            gru_out, _ = self.bigru(packed)
            gru_out, _ = nn.utils.rnn.pad_packed_sequence(
                gru_out, batch_first=True, total_length=T)
        else:
            gru_out, _ = self.bigru(x)
        gru_out      = self.norm(self.dropout(gru_out))
        attn_weights = self.attention(gru_out, lengths)
        action_logits = self.action_head(gru_out)
        anomaly_score = self.anomaly_head(gru_out)
        return action_logits, anomaly_score, attn_weights

model = BiGRUDualHead(
    input_dim=FEAT_DIM, hidden_dim=256,
    num_layers=2, num_classes=NUM_CLASSES, dropout=0.5
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'BiGRU | classes={NUM_CLASSES} | params={total_params:,} | device={DEVICE}')
print('✅ Modèle prêt')

BiGRU | classes=15 | params=2,896,401 | device=cuda
✅ Modèle prêt


In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    for feats, labels, lengths in loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        action_logits, _, _ = model(feats, lengths)
        B, T, C = action_logits.shape
        loss = criterion(action_logits.reshape(B*T, C), labels.reshape(B*T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = action_logits.argmax(dim=-1).reshape(B*T)
        labs  = labels.reshape(B*T)
        mask  = labs >= 0
        all_preds.extend(preds[mask].cpu().numpy())
        all_labels.extend(labs[mask].cpu().numpy())
    f1_macro, f1_weighted, _ = compute_f1(all_preds, all_labels, NUM_CLASSES)
    return total_loss / len(loader), f1_macro, f1_weighted

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for feats, labels, lengths in loader:
            feats, labels = feats.to(device), labels.to(device)
            action_logits, _, _ = model(feats, lengths)
            B, T, C = action_logits.shape
            loss = criterion(action_logits.reshape(B*T, C), labels.reshape(B*T))
            total_loss += loss.item()
            preds = action_logits.argmax(dim=-1).reshape(B*T)
            labs  = labels.reshape(B*T)
            mask  = labs >= 0
            all_preds.extend(preds[mask].cpu().numpy())
            all_labels.extend(labs[mask].cpu().numpy())
    f1_macro, f1_weighted, f1_per_class = compute_f1(
        all_preds, all_labels, NUM_CLASSES)
    return total_loss / len(loader), f1_macro, f1_weighted, f1_per_class

# ── Loss + Optimizer ──────────────────────────────────────
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor.to(DEVICE),
    ignore_index=-1, label_smoothing=0.1
)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# ── Training 50 epochs ────────────────────────────────────
NUM_EPOCHS   = 50
PATIENCE     = 7
best_f1      = 0.0
patience_ctr = 0
history      = []

print(f'Training BiGRU 16 classes | {NUM_EPOCHS} epochs | {DEVICE}')
print(f'{"Epoch":>5} {"Train Loss":>10} {"Train F1":>9} {"Val Loss":>9} {"Val F1":>8} {"Val F1w":>8}')
print('-' * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_f1, train_f1w = train_epoch(
        model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_f1, val_f1w, val_f1_per_class = val_epoch(
        model, val_loader, criterion, DEVICE)
    scheduler.step(val_f1)
    history.append({'epoch': epoch, 'train_loss': train_loss,
                    'train_f1': train_f1, 'val_loss': val_loss, 'val_f1': val_f1})
    print(f'{epoch:>5} {train_loss:>10.4f} {train_f1:>9.4f} '
          f'{val_loss:>9.4f} {val_f1:>8.4f} {val_f1w:>8.4f}')
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'optimizer': optimizer.state_dict(), 'val_f1': val_f1,
            'class_to_idx': class_to_idx, 'idx_to_class': idx_to_class,
            'semantic_groups': SEMANTIC_GROUPS_16, 'num_classes': NUM_CLASSES,
        }, '/kaggle/working/bigru_best_16classes.pth')
        print(f'       ✅ Meilleur modèle (val_f1={val_f1:.4f})')
    if val_f1 <= best_f1 - 0.001:
        patience_ctr += 1
    else:
        patience_ctr = 0
    if patience_ctr >= PATIENCE:
        print(f'\nEarly stopping à epoch {epoch}')
        break

print(f'\nMeilleur val F1 macro : {best_f1:.4f}')

Training BiGRU 16 classes | 50 epochs | cuda
Epoch Train Loss  Train F1  Val Loss   Val F1  Val F1w
------------------------------------------------------------


diagnostic pour notebook prochain

In [7]:
import os

# Re-télécharger train_p1 pour extraire PSR labels
url = 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/f58a5f18-9a71-4443-8a48-e59530df9ee8'
os.system(f'curl -L --retry 3 -o /kaggle/working/train_p1.zip "{url}"')

# Extraire SEULEMENT les PSR CSV
os.system('unzip /kaggle/working/train_p1.zip '
          '"*/PSR_labels_with_errors.csv" '
          '-d /kaggle/working/psr_labels/')

os.system('rm /kaggle/working/train_p1.zip')

# Vérifier
import glob
files = glob.glob('/kaggle/working/psr_labels/**/*with_errors*', recursive=True)
print(f'PSR files : {len(files)}')

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 6076M  100 6076M    0     0  15.7M      0  0:06:25  0:06:25 --:--:-- 16.2M
replace /kaggle/working/psr_labels/01_assy_0_1/PSR_labels_with_errors.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename:  NULL
(EOF or read error, treating as "[N]one" ...)


Archive:  /kaggle/working/train_p1.zip
PSR files : 9


In [9]:
psr_df['is_error'] = psr_df['description'].str.contains(
    'Incorrect|Remove', case=False, na=False).astype(int)
psr_df['frame_num'] = psr_df['frame'].str.replace('.jpg','').astype(int)

errors = psr_df[psr_df['is_error']==1]
normal = psr_df[psr_df['is_error']==0]

print(f'Actions normales : {len(normal)}')
print(f'Erreurs          : {len(errors)}')
print(f'\nErreurs par recording :')
for rec, grp in errors.groupby('recording'):
    print(f'  {rec} : {len(grp)} erreurs')
    for _, row in grp.iterrows():
        print(f'    frame={row["frame_num"]:6d} | {row["description"]}')

Actions normales : 65
Erreurs          : 19

Erreurs par recording :
  01_assy_1_1 : 1 erreurs
    frame=  2102 | Incorrectly installed rear wheel assy
  01_main_0_1 : 5 erreurs
    frame=    78 | Remove rear wheel assy
    frame=   267 | Remove rear rear chassis pin
    frame=   317 | Remove rear chassis
    frame=   317 | Remove front rear chassis pin
    frame=   875 | Remove front rear chassis pin
  02_assy_0_1 : 2 erreurs
    frame=  1143 | Incorrectly installed front chassis
    frame=  1693 | Remove front chassis pin
  02_assy_1_2 : 1 erreurs
    frame=  3563 | Incorrectly installed front wheel assy
  02_main_0_1 : 4 erreurs
    frame=    98 | Remove rear wheel assy
    frame=   399 | Remove rear rear chassis pin
    frame=   522 | Remove rear chassis
    frame=   522 | Remove front rear chassis pin
  04_assy_2_1 : 2 erreurs
    frame=  1880 | Incorrectly installed front bracket
    frame=  1880 | Incorrectly installed front bracket screw
  04_main_0_1 : 4 erreurs
    frame=    

In [8]:
import pandas as pd
import glob

# Lire tous les PSR with errors de train_p1
files = glob.glob('/kaggle/working/psr_labels/**/*with_errors*', recursive=True)
print(f'Fichiers PSR : {len(files)}')

all_psr = []
for f in files:
    rec = f.split('/')[-2]
    df  = pd.read_csv(f, header=None, names=['frame','step_id','description'])
    df['recording'] = rec
    all_psr.append(df)

psr_df = pd.concat(all_psr, ignore_index=True)

print(f'\n=== CONTENU PSR ===')
print(f'Total lignes  : {len(psr_df)}')
print(f'Recordings    : {psr_df["recording"].nunique()}')
print(f'\nPremières lignes :')
print(psr_df.head(10).to_string())

print(f'\n=== TOUTES LES DESCRIPTIONS UNIQUES ===')
for desc in sorted(psr_df['description'].unique()):
    print(f'  {desc}')

Fichiers PSR : 9

=== CONTENU PSR ===
Total lignes  : 84
Recordings    : 9

Premières lignes :
        frame  step_id                     description    recording
0  000085.jpg       32          Remove rear wheel assy  04_main_0_1
1  000326.jpg       20    Remove rear rear chassis pin  04_main_0_1
2  000421.jpg       11             Remove rear chassis  04_main_0_1
3  000421.jpg       17   Remove front rear chassis pin  04_main_0_1
4  001103.jpg       12      Install short rear chassis  04_main_0_1
5  001103.jpg       15  Install front rear chassis pin  04_main_0_1
6  001301.jpg       18   Install rear rear chassis pin  04_main_0_1
7  001728.jpg       30         Install rear wheel assy  04_main_0_1
8  001170.jpg        9            Install rear chassis  04_assy_0_1
9  001170.jpg       15  Install front rear chassis pin  04_assy_0_1

=== TOUTES LES DESCRIPTIONS UNIQUES ===
  Incorrectly installed front bracket
  Incorrectly installed front bracket screw
  Incorrectly installed front chas

In [10]:
import os

BATCH_URLS = {
    'train_p2': 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/5aae5141-cece-41db-9681-47acd5300705',
    'train_p3': 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/4e2ed830-a1fe-40fa-9c5a-2aeae3964616',
    'train_p4': 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/3ff5409e-165f-4354-9fa4-a4fde639a7a6',
    'test_p1' : 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/ec7a4f94-7c1d-497d-b75a-4e694b7a9f7e',
    'test_p2' : 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/d07789b6-83b1-429d-be0c-95edba210757',
    'test_p3' : 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/31f88cb6-a2e7-44df-ac51-5aaa0a148fc3',
}

for batch, url in BATCH_URLS.items():
    zip_path = f'/kaggle/working/{batch}.zip'
    print(f'Téléchargement {batch}...')
    os.system(f'curl -L --retry 3 -o {zip_path} "{url}"')
    os.system(f'unzip -o {zip_path} '
              f'"*/PSR_labels_with_errors.csv" '
              f'-d /kaggle/working/psr_labels/')
    os.system(f'rm {zip_path}')
    print(f'✅ {batch} done')

import glob
files = glob.glob('/kaggle/working/psr_labels/**/*with_errors*', recursive=True)
print(f'\nTotal PSR files : {len(files)}')

Téléchargement train_p2...


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4578M  100 4578M    0     0  20.4M      0  0:03:43  0:03:43 --:--:-- 23.8M


Archive:  /kaggle/working/train_p2.zip
  inflating: /kaggle/working/psr_labels/06_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/06_assy_1_4/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/06_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/07_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/07_assy_2_3/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/07_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/11_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/11_assy_3_3/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/11_main_0_1/PSR_labels_with_errors.csv  
✅ train_p2 done
Téléchargement train_p3...


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4210M  100 4210M    0     0  21.8M      0  0:03:12  0:03:12 --:--:-- 22.5M


Archive:  /kaggle/working/train_p3.zip
  inflating: /kaggle/working/psr_labels/15_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/15_main_3_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/15_main_3_2/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/16_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/16_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/16_main_3_3/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/21_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/21_main_0_1/PSR_labels_with_errors.csv  
✅ train_p3 done
Téléchargement train_p4...


...

Archive:  /kaggle/working/test_p1.zip
  inflating: /kaggle/working/psr_labels/03_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/03_assy_1_3/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/03_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/08_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/08_assy_2_4/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/08_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/09_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/09_assy_3_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/09_main_0_1/PSR_labels_with_errors.csv  
✅ test_p1 done
Téléchargement test_p2...


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5676M  100 5676M    0     0  21.8M      0  0:04:19  0:04:19 --:--:-- 25.9M


Archive:  /kaggle/working/test_p2.zip
  inflating: /kaggle/working/psr_labels/10_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/10_assy_3_2/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/10_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/10_main_3_4/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/12_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/12_assy_3_4/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/12_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/13_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/13_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/13_main_1_3/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/13_main_2_1/PSR_labels_with_errors.csv  
✅ test_p2 done
Téléchargement test_p3...


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9762M  100 9762M    0     0  12.7M      0  0:12:43  0:12:43 --:--:-- 9548k


Archive:  /kaggle/working/test_p3.zip
  inflating: /kaggle/working/psr_labels/17_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/17_assy_1_5/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/17_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/18_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/18_assy_2_5/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/18_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/19_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/19_assy_3_5/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/19_main_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/23_assy_0_1/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/23_assy_1_2/PSR_labels_with_errors.csv  
  inflating: /kaggle/working/psr_labels/23_main_0_1/PSR_labels_with_err

In [13]:
import os
os.system('curl -L -o /kaggle/working/labels.zip "https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/778d7d5f-6631-4cb6-9451-f88c574c7991"')
os.system('unzip -o /kaggle/working/labels.zip -d /kaggle/working/labels/')

import pandas as pd
col_names = ['recording', 'action_id', 'action_name', 'frame_start', 'frame_end']
train_df  = pd.read_csv('/kaggle/working/labels/train.csv', header=None, names=col_names)
val_df    = pd.read_csv('/kaggle/working/labels/val.csv',   header=None, names=col_names)
test_df   = pd.read_csv('/kaggle/working/labels/test.csv',  header=None, names=col_names)
for df in [train_df, val_df, test_df]:
    df['frame_start'] = df['frame_start'].str.replace('.jpg','').astype(int)
    df['frame_end']   = df['frame_end'].str.replace('.jpg','').astype(int)

train_recs = set(train_df['recording'].unique())
val_recs   = set(val_df['recording'].unique())
test_recs  = set(test_df['recording'].unique())

print(f'Train : {len(train_recs)} | Val : {len(val_recs)} | Test : {len(test_recs)}')

# Relevé complet
print(f'\n=== RELEVÉ COMPLET ERREURS ===')
print(f'Total erreurs : {len(errors)}')
print(f'Recordings    : {errors["recording"].nunique()}')
print(f'\n{"Recording":>15} {"Frame":>8} {"Split":>6}  {"Description"}')
print('-' * 75)

for _, row in errors.iterrows():
    rec = row['recording']
    split = 'TRAIN' if rec in train_recs else 'VAL' if rec in val_recs else 'TEST' if rec in test_recs else '???'
    print(f'{rec:>15} {row["frame_num"]:>8} {split:>6}  {row["description"]}')

# Résumé par split
print(f'\n=== RÉSUMÉ PAR SPLIT ===')
for split_name, recs in [('TRAIN', train_recs), ('VAL', val_recs), ('TEST', test_recs)]:
    mask = errors['recording'].isin(recs)
    cnt  = mask.sum()
    recs_with_errors = errors[mask]['recording'].nunique()
    print(f'{split_name:>6} : {cnt:3d} erreurs dans {recs_with_errors} recordings')

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

Archive:  /kaggle/working/labels.zip
  inflating: /kaggle/working/labels/test.csv  
  inflating: /kaggle/working/labels/train.csv  
  inflating: /kaggle/working/labels/val.csv  
Train : 36 | Val : 16 | Test : 32

=== RELEVÉ COMPLET ERREURS ===
Total erreurs : 163
Recordings    : 47

      Recording    Frame  Split  Description
---------------------------------------------------------------------------
    01_assy_1_1     2102  TRAIN  Incorrectly installed rear wheel assy
    01_main_0_1       78  TRAIN  Remove rear wheel assy
    01_main_0_1      267  TRAIN  Remove rear rear chassis pin
    01_main_0_1      317  TRAIN  Remove rear chassis
    01_main_0_1      317  TRAIN  Remove front rear chassis pin
    01_main_0_1      875  TRAIN  Remove front rear chassis pin
    02_assy_0_1     1143  TRAIN  Incorrectly installed front chassis
    02_assy_0_1     1693  TRAIN  Remove front chassis pin
    02_assy_1_2     3563  TRAIN  Incorrectly installed front wheel assy
    02_main_0_1       98  TR

100 67816  100 67816    0     0  69104      0 --:--:-- --:--:-- --:--:-- 69059


In [14]:

# Grouper par type d'erreur
print('=== ERREURS PAR TYPE ===\n')

error_types = errors.groupby('description').size().sort_values(ascending=False)

for desc, cnt in error_types.items():
    recs = errors[errors['description']==desc]['recording'].nunique()
    print(f'{cnt:>4}x | {recs:>3} recordings | {desc}')

print(f'\nTotal types distincts : {len(error_types)}')

=== ERREURS PAR TYPE ===

  32x |  30 recordings | Remove front rear chassis pin
  31x |  31 recordings | Remove rear chassis
  30x |  30 recordings | Remove rear wheel assy
  30x |  29 recordings | Remove rear rear chassis pin
   8x |   8 recordings | Incorrectly installed rear wheel assy
   5x |   4 recordings | Incorrectly installed front bracket screw
   5x |   5 recordings | Incorrectly installed front rear chassis pin
   4x |   4 recordings | Incorrectly installed front wheel assy
   3x |   3 recordings | Remove front chassis pin
   3x |   3 recordings | Incorrectly installed front chassis
   3x |   3 recordings | Incorrectly installed rear rear chassis pin
   2x |   2 recordings | Incorrectly installed front bracket
   2x |   2 recordings | Incorrectly installed short rear chassis
   1x |   1 recordings | Incorrectly installed front chassis pin
   1x |   1 recordings | Remove front bracket
   1x |   1 recordings | Remove front chassis
   1x |   1 recordings | Remove front bracke

In [36]:
# Regarder la séquence complète d'un recording avec erreurs
# Pour comprendre le contexte de "Remove"

import glob, pandas as pd

# Lire PSR_labels.csv (pas with_errors) pour voir séquence normale
files_normal = glob.glob('/kaggle/working/psr_labels/**/*labels.csv', recursive=True)
files_normal = [f for f in files_normal if 'with_errors' not in f and 'raw' not in f]

# Prendre 01_main_0_1 qui a des Remove
for f in files_normal:
    if '01_main_0_1' in f:
        df_normal = pd.read_csv(f, header=None, 
                                names=['frame','step_id','description'])
        print('=== PSR_labels.csv (séquence normale) ===')
        print(df_normal.to_string())
        break

# Comparer avec PSR_labels_with_errors.csv
for f in glob.glob('/kaggle/working/psr_labels/01_main_0_1/*with_errors*'):
    df_errors = pd.read_csv(f, header=None,
                            names=['frame','step_id','description'])
    print('\n=== PSR_labels_with_errors.csv ===')
    print(df_errors.to_string())


=== PSR_labels_with_errors.csv ===
        frame  step_id                     description
0  000078.jpg       32          Remove rear wheel assy
1  000267.jpg       20    Remove rear rear chassis pin
2  000317.jpg       11             Remove rear chassis
3  000317.jpg       17   Remove front rear chassis pin
4  000769.jpg       12      Install short rear chassis
5  000769.jpg       15  Install front rear chassis pin
6  000840.jpg       18   Install rear rear chassis pin
7  000875.jpg       17   Remove front rear chassis pin
8  000917.jpg       15  Install front rear chassis pin
9  001298.jpg       30         Install rear wheel assy


In [44]:
import glob

# Chercher tous les fichiers PSR labels
all_psr_files = glob.glob('/kaggle/working/psr_labels/**/*.csv', recursive=True)
print('Tous les fichiers PSR :')
for f in sorted(all_psr_files):
    print(f'  {f}')

Tous les fichiers PSR :
  /kaggle/working/psr_labels/01_assy_0_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/01_assy_1_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/01_main_0_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/02_assy_0_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/02_assy_1_2/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/02_main_0_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/03_assy_0_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/03_assy_1_3/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/03_main_0_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/04_assy_0_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/04_assy_2_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/04_main_0_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/06_assy_0_1/PSR_labels_with_errors.csv
  /kaggle/working/psr_labels/06_assy_1_4/PSR_labels_with_errors.csv
  /kaggle/working/psr_la

Ça va montrer la séquence complète — si après chaque "Remove" il y a un "Install" du même composant → Remove = correction. Si non → Remove = erreur pure.

In [45]:
import pandas as pd

# Lire 01_main_0_1 with_errors complet
df = pd.read_csv('/kaggle/working/psr_labels/01_main_0_1/PSR_labels_with_errors.csv',
                 header=None, names=['frame','step_id','description'])
df['frame_num'] = df['frame'].str.replace('.jpg','').astype(int)
df = df.sort_values('frame_num')

print('=== 01_main_0_1 séquence complète ===')
print(df.to_string())

=== 01_main_0_1 séquence complète ===
        frame  step_id                     description  frame_num
0  000078.jpg       32          Remove rear wheel assy         78
1  000267.jpg       20    Remove rear rear chassis pin        267
2  000317.jpg       11             Remove rear chassis        317
3  000317.jpg       17   Remove front rear chassis pin        317
4  000769.jpg       12      Install short rear chassis        769
5  000769.jpg       15  Install front rear chassis pin        769
6  000840.jpg       18   Install rear rear chassis pin        840
7  000875.jpg       17   Remove front rear chassis pin        875
8  000917.jpg       15  Install front rear chassis pin        917
9  001298.jpg       30         Install rear wheel assy       1298


frame 78  : Remove rear wheel assy        ← retire roue (erreur antérieure)
frame 267 : Remove rear rear chassis pin  ← retire pin
frame 317 : Remove rear chassis           ← retire chassis
frame 317 : Remove front rear chassis pin ← retire pin
frame 769 : Install short rear chassis    ← réinstalle ✅ correction
frame 769 : Install front rear chassis pin← réinstalle ✅ correction
frame 840 : Install rear rear chassis pin ← réinstalle ✅ correction
frame 875 : Remove front rear chassis pin ← retire encore ← 2ème erreur
frame 917 : Install front rear chassis pin← réinstalle ✅ correction
frame 1298: Install rear wheel assy       ← réinstalle ✅ correction

"Incorrectly installed..." = erreur réelle ✅
→ opérateur a mal installé la pièce
→ frame = moment de l'erreur

"Remove..." = conséquence de l'erreur ✅  
→ opérateur retire ce qu'il a mal installé
→ frame = correction en cours
→ c'est AUSSI anormal car dans séquence normale
  personne ne retire une pièce

In [47]:
import os
os.system('zip -r /kaggle/working/psr_labels_all.zip /kaggle/working/psr_labels/')
os.system('ls -lh /kaggle/working/psr_labels_all.zip')

updating: kaggle/working/psr_labels/ (stored 0%)
updating: kaggle/working/psr_labels/16_main_3_3/ (stored 0%)
updating: kaggle/working/psr_labels/16_main_3_3/PSR_labels_with_errors.csv (deflated 59%)
updating: kaggle/working/psr_labels/27_main_0_1/ (stored 0%)
updating: kaggle/working/psr_labels/27_main_0_1/PSR_labels_with_errors.csv (deflated 59%)
updating: kaggle/working/psr_labels/27_main_3_1/ (stored 0%)
updating: kaggle/working/psr_labels/27_main_3_1/PSR_labels_with_errors.csv (deflated 57%)
updating: kaggle/working/psr_labels/10_main_3_4/ (stored 0%)
updating: kaggle/working/psr_labels/10_main_3_4/PSR_labels_with_errors.csv (deflated 60%)
updating: kaggle/working/psr_labels/06_assy_0_1/ (stored 0%)
updating: kaggle/working/psr_labels/06_assy_0_1/PSR_labels_with_errors.csv (deflated 63%)
updating: kaggle/working/psr_labels/04_main_0_1/ (stored 0%)
updating: kaggle/working/psr_labels/04_main_0_1/PSR_labels_with_errors.csv (deflated 59%)
updating: kaggle/working/psr_labels/11_assy_0

0

Les deux = frames anormales à masquer en training
Les deux = frames à détecter avec Mahalanobis

Séquence normale n'a JAMAIS de "Remove"
→ si BiGRU voit un "Remove" → anomalie garantie
→ Mahalanobis distance = élevée ✅
PIPELINE
Masquer Incorrectly + Remove en training → BiGRU apprend normal
Évaluer Mahalanobis sur ces frames → AUC-ROC

In [9]:
import os, re, random, pickle, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from collections import defaultdict

# ── Chemins ───────────────────────────────────────────────
TRAIN_FEAT_DIR = '/kaggle/input/datasets/hibabou/industrial/data/train/'
VAL_FEAT_DIR   = '/kaggle/input/datasets/hibabou/industrial/data/val/'
TEST_FEAT_DIR  = '/kaggle/input/datasets/hibabou/featuressupp/sf_features_test/'
PREP_DIR       = '/kaggle/input/datasets/hibabou/prepbigru/kaggle/working/'
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FEAT_DIM       = 2304


# ── Semantic groups ───────────────────────────────────────
SEMANTIC_GROUPS_2 = {
    # 0 - NON_CRITIQUE (take + put + pull + check + browse)
    0:0, 2:0, 4:0, 5:0, 8:0, 9:0, 10:0, 11:0, 13:0,
    15:0, 17:0, 18:0, 20:0, 21:0, 23:0, 27:0, 43:0, 65:0,
    12:0, 14:0, 24:0, 35:0, 36:0, 40:0, 42:0, 44:0, 45:0,
    47:0, 48:0, 52:0, 54:0, 57:0, 59:0, 60:0, 62:0, 67:0,
    38:0, 41:0, 46:0, 49:0, 58:0, 61:0, 70:0, 72:0,
    7:0, 29:0, 51:0,
    # 1 - CRITIQUE (fit + plug + tighten + loosen + align)
    1:1, 3:1, 6:1, 16:1, 19:1, 22:1, 25:1, 26:1, 28:1,
    30:1, 31:1, 32:1, 33:1, 34:1, 39:1, 50:1, 53:1,
    55:1, 56:1, 63:1, 64:1, 68:1, 69:1, 71:1, 73:1, 74:1,
}

NUM_CLASSES    = 2
CLASS_NAMES_2  = {0: 'NON_CRITIQUE', 1: 'CRITIQUE'}

# ── Labels CSV ────────────────────────────────────────────
os.system('curl -L -o /kaggle/working/labels.zip "https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/778d7d5f-6631-4cb6-9451-f88c574c7991"')
os.system('unzip -o /kaggle/working/labels.zip -d /kaggle/working/labels/')

col_names = ['recording', 'action_id', 'action_name', 'frame_start', 'frame_end']
train_df  = pd.read_csv('/kaggle/working/labels/train.csv', header=None, names=col_names)
val_df    = pd.read_csv('/kaggle/working/labels/val.csv',   header=None, names=col_names)
test_df   = pd.read_csv('/kaggle/working/labels/test.csv',  header=None, names=col_names)

for df in [train_df, val_df, test_df]:
    df['frame_start'] = df['frame_start'].str.replace('.jpg','').astype(int)
    df['frame_end']   = df['frame_end'].str.replace('.jpg','').astype(int)
    df['label2']      = df['action_id'].map(SEMANTIC_GROUPS_2)

print(f'Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)} ✅')

# ── Charger pkl ───────────────────────────────────────────
with open(f'{PREP_DIR}scaler.pkl',            'rb') as f: scaler            = pickle.load(f)
with open(f'{PREP_DIR}stride_map_train.pkl',  'rb') as f: stride_map_train  = pickle.load(f)
with open(f'{PREP_DIR}stride_map_val.pkl',    'rb') as f: stride_map_val    = pickle.load(f)
with open(f'{PREP_DIR}rec_to_file_train.pkl', 'rb') as f: rec_to_file_train = pickle.load(f)
with open(f'{PREP_DIR}rec_to_file_val.pkl',   'rb') as f: rec_to_file_val   = pickle.load(f)

# ── Stride map test ───────────────────────────────────────
rec_to_file_test = {}
for f in os.listdir(TEST_FEAT_DIR):
    if f.endswith('.npy'):
        rec = re.sub(r'^test_p\d+_', '', f).replace('_features.npy', '')
        rec_to_file_test[rec] = os.path.join(TEST_FEAT_DIR, f)

stride_map_test = {}
for rec, path in rec_to_file_test.items():
    sub = test_df[test_df['recording'] == rec]
    if len(sub) == 0:
        continue
    arr    = np.load(path)
    f_min  = sub['frame_start'].min()
    f_max  = sub['frame_end'].max()
    n_feat = arr.shape[0]
    total  = f_max - f_min
    stride = round((total - 32) / (n_feat - 1)) if n_feat > 1 else 1
    stride = max(1, stride)
    stride_map_test[rec] = {'stride': stride, 'f_min': f_min, 'n_feats': n_feat}

print(f'Stride maps → Train:{len(stride_map_train)} Val:{len(stride_map_val)} Test:{len(stride_map_test)} ✅')

# ── PSR Error frames ──────────────────────────────────────
psr_files = glob.glob('/kaggle/input/datasets/hibabou/psr-labels/kaggle/working/psr_labels/**/*with_errors*', recursive=True)
all_psr   = []
for f in psr_files:
    rec = f.split('/')[-2]
    df  = pd.read_csv(f, header=None, names=['frame','step_id','description'])
    df['recording'] = rec
    all_psr.append(df)

psr_df = pd.concat(all_psr, ignore_index=True)
psr_df['is_error']  = psr_df['description'].str.contains(
    'Incorrect|Remove', case=False, na=False).astype(int)
psr_df['frame_num'] = psr_df['frame'].str.replace('.jpg','').astype(int)

# Dictionnaire erreurs par recording
error_frames_dict = defaultdict(list)
for _, row in psr_df[psr_df['is_error']==1].iterrows():
    error_frames_dict[row['recording']].append(row['frame_num'])

print(f'PSR files : {len(psr_files)}')
print(f'Erreurs totales : {psr_df["is_error"].sum()}')
print(f'Recordings avec erreurs : {len(error_frames_dict)}')

# ── Fonctions utilitaires ─────────────────────────────────
def get_feat_indices(frame_start, frame_end, f_min, n_feats, stride):
    i0 = max(0, round((frame_start - f_min) / stride))
    i1 = min(n_feats, round((frame_end   - f_min) / stride))
    return i0, i1

def compute_f1_metrics(all_preds, all_labels, num_classes):
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    mask       = all_labels >= 0
    f1_macro    = f1_score(all_labels[mask], all_preds[mask],
                           average='macro',    zero_division=0)
    f1_weighted = f1_score(all_labels[mask], all_preds[mask],
                           average='weighted', zero_division=0)
    f1_per      = f1_score(all_labels[mask], all_preds[mask],
                           average=None, zero_division=0,
                           labels=list(range(num_classes)))
    return f1_macro, f1_weighted, f1_per

print(f'Device : {DEVICE} ✅')

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 67816  100 67816    0     0  79871      0 --:--:-- --:--:-- --:--:-- 79877


Archive:  /kaggle/working/labels.zip
  inflating: /kaggle/working/labels/test.csv  
  inflating: /kaggle/working/labels/train.csv  
  inflating: /kaggle/working/labels/val.csv  
Train : 3667 | Val : 1928 | Test : 3678 ✅
Stride maps → Train:36 Val:16 Test:32 ✅
PSR files : 68
Erreurs totales : 163
Recordings avec erreurs : 47
Device : cuda ✅


In [17]:
# ── Dataset avec masquage frames erreur ───────────────────
class IndustRealDatasetClean(Dataset):
    def __init__(self, dfs, rec_to_files, stride_maps, scaler,
                 semantic_groups, error_frames_dict=None,
                 clip_val=5.0, seq_len=128, step=64,
                 is_train=False, oversample_cls=1,
                 oversample_factor=2, undersample_cls=0,
                 undersample_ratio=0.35, error_window=30):
        self.samples  = []
        self.is_train = is_train

        if not isinstance(dfs, list):
            dfs          = [dfs]
            rec_to_files = [rec_to_files]
            stride_maps  = [stride_maps]

        total_masked = 0

        for df, rec_to_file, stride_map in zip(dfs, rec_to_files, stride_maps):
            for rec, info in stride_map.items():
                sub = df[df['recording'] == rec]
                if len(sub) == 0:
                    continue

                arr = np.load(rec_to_file[rec]).astype(np.float32)
                arr = scaler.transform(arr).astype(np.float32)
                arr = np.clip(arr, -5.0, 5.0)

                # Labels action
                labels = np.full(arr.shape[0], -1, dtype=np.int64)
                for _, row in sub.iterrows():
                    if row['action_id'] not in semantic_groups:
                        continue
                    i0, i1 = get_feat_indices(
                        row['frame_start'], row['frame_end'],
                        info['f_min'], info['n_feats'], info['stride']
                    )
                    labels[i0:i1] = semantic_groups[row['action_id']]

                # Masquer frames d'erreur PSR → label = -1
                if error_frames_dict and rec in error_frames_dict:
                    for err_frame in error_frames_dict[rec]:
                        err_idx = round((err_frame - info['f_min']) / info['stride'])
                        err_idx = max(0, min(err_idx, info['n_feats']-1))
                        start   = max(0, err_idx - error_window)
                        end     = min(info['n_feats'], err_idx + error_window)
                        n_before = (labels[start:end] >= 0).sum()
                        labels[start:end] = -1
                        total_masked += n_before

                # Découper en fenêtres
                for start in range(0, len(arr) - seq_len + 1, step):
                    end          = start + seq_len
                    chunk_feats  = arr[start:end]
                    chunk_labels = labels[start:end]
                    labeled      = chunk_labels[chunk_labels >= 0]

                    if len(labeled) < seq_len // 4:
                        continue

                    dominant = np.bincount(labeled, minlength=NUM_CLASSES).argmax()

                    # Undersampling
                    if is_train and dominant == undersample_cls:
                        if np.random.random() > undersample_ratio:
                            continue

                    self.samples.append({
                        'features' : torch.tensor(chunk_feats,  dtype=torch.float32),
                        'labels'   : torch.tensor(chunk_labels, dtype=torch.long),
                        'dominant' : dominant
                    })

        # Oversampling
        if is_train and oversample_factor > 1:
            verif = [s for s in self.samples if s['dominant'] == oversample_cls]
            for _ in range(oversample_factor - 1):
                self.samples.extend(verif)
            random.shuffle(self.samples)

        # Stats
        counts = defaultdict(int)
        for s in self.samples:
            counts[s['dominant']] += 1
        total = len(self.samples)

        print(f'\nDataset : {total} segments | train={is_train}')
        print(f'Frames masquées (erreurs PSR) : {total_masked}')
        for cls in range(NUM_CLASSES):
            cnt = counts[cls]
            print(f'  {CLASS_NAMES_2[cls]:15s} : {cnt:4d} ({cnt/total:.1%})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        feats  = self.samples[idx]['features'].clone()
        labels = self.samples[idx]['labels'].clone()

        if self.is_train:
            if random.random() > 0.5:
                feats  = feats.flip(0)
                labels = labels.flip(0)
            if random.random() > 0.5:
                feats = feats + torch.randn_like(feats) * 0.05
            if random.random() > 0.7:
                mask_len   = random.randint(5, 20)
                mask_start = random.randint(0, max(0, len(feats) - mask_len))
                feats[mask_start:mask_start+mask_len] = 0.0

        return feats, labels

def collate_fn(batch):
    feats, labels = zip(*batch)
    return torch.stack(feats), torch.stack(labels), \
           torch.tensor([f.shape[0] for f in feats])

# ── Instancier ────────────────────────────────────────────
SEQ_LEN    = 128
STEP       = 64
BATCH_SIZE = 16

 
train_dataset = IndustRealDatasetClean(
    dfs             = [train_df,          test_df],
    rec_to_files    = [rec_to_file_train, rec_to_file_test],
    stride_maps     = [stride_map_train,  stride_map_test],
    scaler          = scaler,
    semantic_groups = SEMANTIC_GROUPS_2,
    error_frames_dict = error_frames_dict,
    seq_len         = SEQ_LEN,
    step            = STEP,
    is_train        = True,
    oversample_cls    = 0,
    oversample_factor = 1,   # pas d'oversample
    undersample_cls   = 1,
    undersample_ratio = 0.8, # garder 80% CRITIQUE
    error_window      = 30
)
val_dataset = IndustRealDatasetClean(
    dfs               = val_df,
    rec_to_files      = rec_to_file_val,
    stride_maps       = stride_map_val,
    scaler            = scaler,
    semantic_groups   = SEMANTIC_GROUPS_2,  # cohérent avec NUM_CLASSES=2
    error_frames_dict = None,
    seq_len           = SEQ_LEN,
    step              = STEP,
    is_train          = False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)

print(f'\nTrain loader : {len(train_loader)} batches')
print(f'Val   loader : {len(val_loader)} batches')

# ── Class weights ─────────────────────────────────────────
all_labels_train = []
for s in train_dataset.samples:
    labs = s['labels'].numpy()
    all_labels_train.extend(labs[labs >= 0].tolist())

classes = np.array([0, 1])            # 2 classes : NON_CRITIQUE, CRITIQUE
weights = compute_class_weight('balanced', classes=classes,
                                y=np.array(all_labels_train))
weights = np.clip(weights, 0, 5.0)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

print(f'\nClass weights :')
for i, w in enumerate(weights):
    print(f'  {CLASS_NAMES_2[i]:15s} : {w:.3f}')
print('✅ Dataset prêt')


Dataset : 1870 segments | train=True
Frames masquées (erreurs PSR) : 5311
  NON_CRITIQUE    : 1050 (56.1%)
  CRITIQUE        :  820 (43.9%)

Dataset : 270 segments | train=False
Frames masquées (erreurs PSR) : 0
  NON_CRITIQUE    :  159 (58.9%)
  CRITIQUE        :  111 (41.1%)

Train loader : 117 batches
Val   loader : 17 batches

Class weights :
  NON_CRITIQUE    : 0.932
  CRITIQUE        : 1.079
✅ Dataset prêt


In [16]:
# Voir combien de frames CRITIQUE masquées vs NON_CRITIQUE
import numpy as np

critique_total   = 0
critique_masked  = 0
non_crit_total   = 0
non_crit_masked  = 0

for rec, info in stride_map_train.items():
    sub = train_df[train_df['recording'] == rec]
    if len(sub) == 0:
        continue
    arr    = np.load(rec_to_file_train[rec])
    labels = np.full(arr.shape[0], -1, dtype=np.int64)
    
    for _, row in sub.iterrows():
        if row['action_id'] not in SEMANTIC_GROUPS_2:
            continue
        i0, i1 = get_feat_indices(
            row['frame_start'], row['frame_end'],
            info['f_min'], info['n_feats'], info['stride']
        )
        labels[i0:i1] = SEMANTIC_GROUPS_2[row['action_id']]
    
    # Mask erreurs
    error_mask = np.zeros(arr.shape[0], dtype=bool)
    if rec in error_frames_dict:
        for err_frame in error_frames_dict[rec]:
            err_idx = round((err_frame - info['f_min']) / info['stride'])
            err_idx = max(0, min(err_idx, info['n_feats']-1))
            start   = max(0, err_idx - 30)
            end     = min(info['n_feats'], err_idx + 30)
            error_mask[start:end] = True
    
    # Compter
    for i in range(len(labels)):
        if labels[i] == 1:
            critique_total += 1
            if error_mask[i]:
                critique_masked += 1
        elif labels[i] == 0:
            non_crit_total += 1
            if error_mask[i]:
                non_crit_masked += 1

print(f'CRITIQUE    total={critique_total} | masquées={critique_masked} ({critique_masked/critique_total:.1%})')
print(f'NON_CRITIQUE total={non_crit_total} | masquées={non_crit_masked} ({non_crit_masked/non_crit_total:.1%})')

CRITIQUE    total=18727 | masquées=1171 (6.3%)
NON_CRITIQUE total=20225 | masquées=1384 (6.8%)


Class weights proches de 1.0 → 
dataset naturellement équilibré après masquage
→ pas besoin de beaucoup de correction
→ modèle va apprendre de manière équilibrée ✅

BiGRU → apprend 3 classes sémantiques (F1=0.46)
Mahalanobis → détecte si frame = normale ou anormale
PSR labels → évalue si anormale = erreur ou correction

→ pipeline complet sans changer les classes BiGRU

In [18]:
# ── Modèle BiGRU ──────────────────────────────────────────
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, gru_out, lengths=None):
        scores = self.attn(gru_out).squeeze(-1)
        if lengths is not None:
            for i, l in enumerate(lengths):
                if l < scores.shape[1]:
                    scores[i, l:] = float('-inf')
        return torch.softmax(scores, dim=1)


class ResidualBlock(nn.Module):
    """Bloc résiduel pour stabiliser l'apprentissage"""
    def __init__(self, dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.block(x))


class BiGRUDualHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers,
                 num_classes, dropout=0.4):
        super().__init__()

        # Input projection avec résiduel
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            ResidualBlock(hidden_dim, dropout)
        )

        # BiGRU stack
        self.bigru = nn.GRU(
            input_size    = hidden_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if num_layers > 1 else 0.0
        )

        self.norm      = nn.LayerNorm(hidden_dim * 2)
        self.dropout   = nn.Dropout(dropout)
        self.attention = TemporalAttention(hidden_dim)

        # Action head — profond avec résiduel
        self.action_proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.action_residual = ResidualBlock(hidden_dim, dropout / 2)
        self.action_out = nn.Linear(hidden_dim, num_classes)

        # Anomaly head
        self.anomaly_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.GELU(),
            nn.Linear(hidden_dim // 4, 1),
            nn.Sigmoid()
        )

        # Init weights
        self._init_weights()

    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)

    def forward(self, x, lengths=None):
        B, T, _ = x.shape

        # Input projection
        x = self.input_proj(x)

        # BiGRU
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            gru_out, _ = self.bigru(packed)
            gru_out, _ = nn.utils.rnn.pad_packed_sequence(
                gru_out, batch_first=True, total_length=T)
        else:
            gru_out, _ = self.bigru(x)

        # Norm + dropout
        gru_out = self.norm(self.dropout(gru_out))

        # Attention
        attn_weights = self.attention(gru_out, lengths)

        # Action head
        action_feat   = self.action_proj(gru_out)
        action_feat   = self.action_residual(action_feat)
        action_logits = self.action_out(action_feat)

        # Anomaly head
        anomaly_score = self.anomaly_head(gru_out)

        return action_logits, anomaly_score, attn_weights, gru_out

# ── Instancier ────────────────────────────────────────────
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT    = 0.4

model = BiGRUDualHead(
    input_dim   = FEAT_DIM,
    hidden_dim  = HIDDEN_DIM,
    num_layers  = NUM_LAYERS,
    num_classes = NUM_CLASSES,
    dropout     = DROPOUT
).to(DEVICE)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'BiGRU {NUM_CLASSES} classes')
print(f'  Hidden     : {HIDDEN_DIM}×2 bidirectionnel')
print(f'  Layers     : {NUM_LAYERS}')
print(f'  Classes    : {NUM_CLASSES}')
print(f'  Params     : {params:,}')
print(f'  Residual   : ✅')
print(f'  Attention  : ✅')
print(f'  Device     : {DEVICE}')
print('✅ Modèle prêt')

BiGRU 2 classes
  Hidden     : 512×2 bidirectionnel
  Layers     : 2
  Classes    : 2
  Params     : 11,462,660
  Residual   : ✅
  Attention  : ✅
  Device     : cuda
✅ Modèle prêt


In [20]:
# Recalculer class weights pour 2 classes
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import torch

all_labels_train = []
for s in train_dataset.samples:
    labs = s['labels'].numpy()
    all_labels_train.extend(labs[labs >= 0].tolist())

classes = np.array([0, 1])
weights = compute_class_weight('balanced', classes=classes,
                                y=np.array(all_labels_train))
weights = np.clip(weights, 0, 5.0)
class_weights_tensor_3 = torch.tensor(weights, dtype=torch.float32)

print(f'Class weights :')
for i, w in enumerate(weights):
    print(f'  {CLASS_NAMES_2[i]:15s} : {w:.3f}')

Class weights :
  NON_CRITIQUE    : 0.932
  CRITIQUE        : 1.079


In [21]:
# ── Training functions ────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for feats, labels, lengths in loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()

        action_logits, _, _, _ = model(feats, lengths)
        B, T, C = action_logits.shape
        loss = criterion(
            action_logits.reshape(B*T, C),
            labels.reshape(B*T)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = action_logits.argmax(dim=-1).reshape(B*T)
        labs  = labels.reshape(B*T)
        mask  = labs >= 0
        all_preds.extend(preds[mask].cpu().numpy())
        all_labels.extend(labs[mask].cpu().numpy())

    f1_macro, f1_weighted, _ = compute_f1_metrics(
        all_preds, all_labels, NUM_CLASSES)
    return total_loss / len(loader), f1_macro, f1_weighted


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for feats, labels, lengths in loader:
            feats, labels = feats.to(device), labels.to(device)
            action_logits, _, _, _ = model(feats, lengths)
            B, T, C = action_logits.shape
            loss = criterion(
                action_logits.reshape(B*T, C),
                labels.reshape(B*T)
            )
            total_loss += loss.item()
            preds = action_logits.argmax(dim=-1).reshape(B*T)
            labs  = labels.reshape(B*T)
            mask  = labs >= 0
            all_preds.extend(preds[mask].cpu().numpy())
            all_labels.extend(labs[mask].cpu().numpy())

    f1_macro, f1_weighted, f1_per = compute_f1_metrics(
        all_preds, all_labels, NUM_CLASSES)
    return total_loss / len(loader), f1_macro, f1_weighted, f1_per


# ── Loss + Optimizer ──────────────────────────────────────
criterion = nn.CrossEntropyLoss(
    weight          = class_weights_tensor_3.to(DEVICE),
    ignore_index    = -1,
    label_smoothing = 0.1
)
optimizer = optim.AdamW(
    model.parameters(),
    lr           = 2e-4,
    weight_decay = 1e-3,
    betas        = (0.9, 0.999)
)
scheduler = ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5,
    patience=4, min_lr=1e-6
)

# ── Training loop ─────────────────────────────────────────
NUM_EPOCHS   = 60
PATIENCE     = 12
best_f1      = 0.0
patience_ctr = 0

CLASS_NAMES = CLASS_NAMES_2

print(f'Training BiGRU {NUM_CLASSES} classes | train+test | {DEVICE}')
print(f'{"Ep":>4} {"TLoss":>8} {"TF1":>7} {"VLoss":>8} {"VF1":>7} '
      f'{"NON_C":>7} {"CRIT":>7}')
print('-' * 55)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_f1, _        = train_epoch(
        model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_f1, _, val_f1_pc = val_epoch(
        model, val_loader, criterion, DEVICE)

    scheduler.step(val_f1)

    nc   = val_f1_pc[0] if len(val_f1_pc) > 0 else 0
    crit = val_f1_pc[1] if len(val_f1_pc) > 1 else 0

    print(f'{epoch:>4} {train_loss:>8.4f} {train_f1:>7.4f} '
          f'{val_loss:>8.4f} {val_f1:>7.4f} '
          f'{nc:>7.4f} {crit:>7.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({
            'epoch'           : epoch,
            'model_state'     : model.state_dict(),
            'optimizer'       : optimizer.state_dict(),
            'val_f1'          : val_f1,
            'val_f1_per_class': val_f1_pc.tolist(),
            'semantic_groups' : SEMANTIC_GROUPS_2,
            'num_classes'     : NUM_CLASSES,
            'hidden_dim'      : HIDDEN_DIM,
            'class_names'     : CLASS_NAMES_2,
        }, '/kaggle/working/bigru_2classes_best.pth')
        print(f'  ✅ Saved (val_f1={val_f1:.4f} | '
              f'NON_C={nc:.3f} CRIT={crit:.3f})')

    patience_ctr = 0 if val_f1 > best_f1 - 0.001 else patience_ctr + 1
    if patience_ctr >= PATIENCE:
        print(f'\nEarly stopping epoch {epoch}')
        break

print(f'\nMeilleur val F1 macro : {best_f1:.4f}')

Training BiGRU 2 classes | train+test | cuda
  Ep    TLoss     TF1    VLoss     VF1   NON_C    CRIT
-------------------------------------------------------
   1   0.7883  0.5194   0.6702  0.5576  0.4711  0.6440
  ✅ Saved (val_f1=0.5576 | NON_C=0.471 CRIT=0.644)
   2   0.6900  0.5634   0.6499  0.6317  0.6137  0.6496
  ✅ Saved (val_f1=0.6317 | NON_C=0.614 CRIT=0.650)
   3   0.6739  0.5896   0.6474  0.6343  0.7051  0.5635
  ✅ Saved (val_f1=0.6343 | NON_C=0.705 CRIT=0.564)
   4   0.6543  0.6192   0.6424  0.6367  0.6544  0.6189
  ✅ Saved (val_f1=0.6367 | NON_C=0.654 CRIT=0.619)
   5   0.6331  0.6510   0.6819  0.6288  0.6989  0.5588
   6   0.6053  0.6898   0.7335  0.5386  0.4661  0.6111
   7   0.5882  0.7058   0.6574  0.6473  0.7208  0.5738
  ✅ Saved (val_f1=0.6473 | NON_C=0.721 CRIT=0.574)
   8   0.5564  0.7405   0.6436  0.6626  0.7011  0.6242
  ✅ Saved (val_f1=0.6626 | NON_C=0.701 CRIT=0.624)
   9   0.5375  0.7591   0.6931  0.6165  0.6205  0.6124
  10   0.5192  0.7732   0.7037  0.6110  0.5


CRITIQUE = actions d'assemblage précis :

fit_*, plug_*, tighten_*, loosen_*, align_*
→ actions où une erreur a des conséquences graves
→ mal serrer un écrou = danger
→ mal insérer une pièce = assemblage défectueux
→ cobot doit surveiller intensément ces actions
NON_CRITIQUE = actions de manipulation et vérification :

take_*, put_*, pull_*, check_*, browse_*
→ actions préparatoires ou de contrôle
→ prendre une mauvaise pièce = détectable visuellement
→ vérifier une instruction = pas dangereux
→ cobot peut surveiller moins intensément
Valeur industrielle :

Cobot sait maintenant :
→ "opérateur est en phase CRITIQUE"
  → augmenter surveillance
  → Mahalanobis plus sensible
  → seuil anomalie plus bas

→ "opérateur est en phase NON_CRITIQUE"
  → surveillance normale
  → tolérance plus haute
C'est plus intelligent que 3 classes car correspond exactement au besoin cobot — surveiller plus les phases critiques d'assemblage.




In [23]:
import zipfile, os

checkpoint_path = "/kaggle/working/bigru_2classes_best.pth"
zip_output      = "/kaggle/working/bigru_2classes_best.zip"

with zipfile.ZipFile(zip_output, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(checkpoint_path, arcname="bigru_2classes_best.pth")

print(f"✅ Zippé : {zip_output}")
print(f"📦 Taille : {os.path.getsize(zip_output) / 1024 / 1024:.2f} MB")

from IPython.display import FileLink
FileLink(zip_output)

✅ Zippé : /kaggle/working/bigru_2classes_best.zip
📦 Taille : 115.14 MB


/kaggle/working/bigru_2classes_best.zip